# 04j Extraction, batch j: ten studies coded from their PDFs and two full-text exclusions

Design log.
- Batch j of notebook 04 (plan v2 B4), coded under the frozen schema (schema/record_schema.yaml, v1) and the coding rules, as batches a to i were. The ten reports: LongPIBench (2608.28411), Universal Defenses for Tool-Integrated LLM Agents (2609.16098), Defusing Explosive Prompts (2609.22510), Readable Before Actionable (2610.05295), TrustMI (2610.06064), Policy-Verified Self-Training (2610.19372), The Model Knows Better (medical multi-agent systems, 2610.22594), PIPETTE (2611.24362), Staying on Task (CBC, EMNLP 2026) and Real User Instruction (Preprints.org).
- Each record's extraction_scope names the tables and rows coded. As before, detectors, sanitisers, filters and system-level designs are left out and the scope says so (amendment 4): PromptLocate and the detectors in LongPIBench, the ATF and NTR combinations in the universal-defences study, Prompt Guard 2, PIGuard, PIShield and DeFuse in the explosive-prompt study, the gate-on rows of PVST, PromptGuard 2, DataSentinel, PromptArmor and the paper's own checker in the medical study, PromptGuard, PIGuard, DataSentinel, MELON and CaMeL in PIPETTE, and PromptArmor in CBC. Direct-injection columns are left out (CBC Table 1, the universal-defences DPI results).
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - Steering edits whose direction is fitted from examples (Readable Before Actionable) are fixed inference rules with axis 1 internal_intervention and training fields not_applicable; TrustMI's steering matrices are trained with BiPO, so they are coded as preference optimisation with the training fields set.
  - In the explosive-prompt study the main-text SecAlign cell uses a different denominator from its baseline, so the appendix table with one denominator (Table 24) is coded.
  - PVST is coded with the runtime gate off, so only the trained agent is measured; its violating-trajectory rate is a proxy outcome beside ASR.
  - In the medical study each system and model pair (single model, MDAgents, MedAgents) is its own base.
  - CBC reports defence success rates (higher is better); its training-based baselines were trained by the CBC authors with the released data.
  - The universal-defences study's OTSR is coded as a proxy for utility under attack because the paper does not say whether it is measured in attacked runs.
  - RUI's benign performance preservation is judged by another LLM against the undefended model's output and is coded as percent_of_reference.
  - Model names printed in the papers are kept as printed, including closed models used as rows (GPT-4o and GPT-4.1 rows in LongPIBench's attack tables, GPT-3.5 and GPT-4 in the universal-defences study, Gemini 3 Flash, Gemini 3.1 Pro and GPT-5.4 in RUI).
- Full-text exclusions found while coding: ResumeShield (2609.20188) evaluates only a rule-based simulated screener, which by construction obeys directives only outside its fence, so no language model is measured; it is excluded as survey, position or other. CoDA (2610.15177) enforces task policies of authorised tools, arguments and URLs during decoding, which is coded as system level, like a tool allowlist. Both reverse the includes recorded at full-text check in notebook 03. Stage 2b checks both anchors in the PDFs and carries the decisions into schema/fulltext_decisions.csv, schema/screening_decisions.csv, reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as earlier batches did. The number of included studies falls from 128 to 126.
- Outputs: reports/records/ (ten files), reports/extraction_04j_check.csv, reports/extraction_04j_summary.csv, reports/extraction_batches.csv (rows for batch j added, earlier batches kept), reports/decision_04j.txt.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 10 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`longpibench_arxiv2026` (2608.28411)

In [ ]:
%%writefile reports/records/longpibench_arxiv2026.yaml
report:
  id: longpibench_arxiv2026
  slug: '2608.28411'
  title: 'LongPIBench: A Long-Context Benchmark for Prompt Injection'
  first_author: Yupei Liu
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2608.28411'
  version_read: arXiv v1
  first_date: '2026-08-28'
  read: F
  role: benchmark
  extraction_scope: 'Tables 4 and 13 (with-attack rows) and Table 5 (OPI, InjecAgent and AgentDojo rows): No defense, Instructional, Delimiters, Sandwich, SecInfer and MetaSecAlign 8B on Llama-3.1-8B-Instruct; the Section
    5.4 claim (PromptLocate runs DataSentinel detection and span removal and the detectors of Tables 3 and 12 are left out; the without-attack rows measure the natural rate of the attacker goal, not utility)'
defences:
- id: instr
  name: Instructional prevention
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: long document (thousands to tens of thousands of tokens) with the injected prompt inside, task instruction in the prompt
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: We benchmark state-of-the-art prevention defenses
- id: delim
  name: Delimiter-based isolation
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: long document (thousands to tens of thousands of tokens) with the injected prompt inside, task instruction in the prompt
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: We benchmark state-of-the-art prevention defenses
- id: sand
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: long document (thousands to tens of thousands of tokens) with the injected prompt inside, task instruction in the prompt
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: We benchmark state-of-the-art prevention defenses
- id: secinfer
  name: 'SecInfer (inference-time scaling: several responses under different system prompts, target-task-guided aggregation)'
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: long document (thousands to tens of thousands of tokens) with the injected prompt inside, task instruction in the prompt
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: We benchmark state-of-the-art prevention defenses
- id: msa
  name: MetaSecAlign 8B (released checkpoint)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: long document (thousands to tens of thousands of tokens) with the injected prompt inside, task instruction in the prompt
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: We benchmark state-of-the-art prevention defenses
evaluations:
- id: ev_syn_paper_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'LongPIBench synthetic, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_syn_paper_instr
  defence_id: instr
  benchmark: 'LongPIBench synthetic, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_paper_delim
  defence_id: delim
  benchmark: 'LongPIBench synthetic, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_paper_sand
  defence_id: sand
  benchmark: 'LongPIBench synthetic, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_paper_secinfer
  defence_id: secinfer
  benchmark: 'LongPIBench synthetic, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_paper_msa
  defence_id: msa
  benchmark: 'LongPIBench synthetic, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_resume_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'LongPIBench synthetic, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_syn_resume_instr
  defence_id: instr
  benchmark: 'LongPIBench synthetic, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_resume_delim
  defence_id: delim
  benchmark: 'LongPIBench synthetic, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_resume_sand
  defence_id: sand
  benchmark: 'LongPIBench synthetic, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_resume_secinfer
  defence_id: secinfer
  benchmark: 'LongPIBench synthetic, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_resume_msa
  defence_id: msa
  benchmark: 'LongPIBench synthetic, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_email_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'LongPIBench synthetic, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_syn_email_instr
  defence_id: instr
  benchmark: 'LongPIBench synthetic, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_email_delim
  defence_id: delim
  benchmark: 'LongPIBench synthetic, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_email_sand
  defence_id: sand
  benchmark: 'LongPIBench synthetic, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_email_secinfer
  defence_id: secinfer
  benchmark: 'LongPIBench synthetic, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_email_msa
  defence_id: msa
  benchmark: 'LongPIBench synthetic, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_code_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'LongPIBench synthetic, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_syn_code_instr
  defence_id: instr
  benchmark: 'LongPIBench synthetic, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_code_delim
  defence_id: delim
  benchmark: 'LongPIBench synthetic, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_code_sand
  defence_id: sand
  benchmark: 'LongPIBench synthetic, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_code_secinfer
  defence_id: secinfer
  benchmark: 'LongPIBench synthetic, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_syn_code_msa
  defence_id: msa
  benchmark: 'LongPIBench synthetic, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_real_paper_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'LongPIBench real-world, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_real_paper_instr
  defence_id: instr
  benchmark: 'LongPIBench real-world, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_paper_delim
  defence_id: delim
  benchmark: 'LongPIBench real-world, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_paper_sand
  defence_id: sand
  benchmark: 'LongPIBench real-world, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_paper_secinfer
  defence_id: secinfer
  benchmark: 'LongPIBench real-world, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_paper_msa
  defence_id: msa
  benchmark: 'LongPIBench real-world, paper review (goal: rating 8 or 9), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_resume_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'LongPIBench real-world, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_real_resume_instr
  defence_id: instr
  benchmark: 'LongPIBench real-world, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_resume_delim
  defence_id: delim
  benchmark: 'LongPIBench real-world, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_resume_sand
  defence_id: sand
  benchmark: 'LongPIBench real-world, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_resume_secinfer
  defence_id: secinfer
  benchmark: 'LongPIBench real-world, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_resume_msa
  defence_id: msa
  benchmark: 'LongPIBench real-world, resume screening (goal: shortlist), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_email_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'LongPIBench real-world, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_real_email_instr
  defence_id: instr
  benchmark: 'LongPIBench real-world, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_email_delim
  defence_id: delim
  benchmark: 'LongPIBench real-world, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_email_sand
  defence_id: sand
  benchmark: 'LongPIBench real-world, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_email_secinfer
  defence_id: secinfer
  benchmark: 'LongPIBench real-world, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_email_msa
  defence_id: msa
  benchmark: 'LongPIBench real-world, email summary (goal: attacker link), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_code_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'LongPIBench real-world, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_real_code_instr
  defence_id: instr
  benchmark: 'LongPIBench real-world, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_code_delim
  defence_id: delim
  benchmark: 'LongPIBench real-world, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_code_sand
  defence_id: sand
  benchmark: 'LongPIBench real-world, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_code_secinfer
  defence_id: secinfer
  benchmark: 'LongPIBench real-world, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_real_code_msa
  defence_id: msa
  benchmark: 'LongPIBench real-world, code review (goal: approve the change), Authority spoof injection'
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 18
  decoding: not reported
- id: ev_t5_opi_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: OPI (Liu et al. 2024), run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_t5_opi_instr
  defence_id: instr
  benchmark: OPI (Liu et al. 2024), run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_opi_delim
  defence_id: delim
  benchmark: OPI (Liu et al. 2024), run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_opi_sand
  defence_id: sand
  benchmark: OPI (Liu et al. 2024), run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_opi_secinfer
  defence_id: secinfer
  benchmark: OPI (Liu et al. 2024), run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_opi_msa
  defence_id: msa
  benchmark: OPI (Liu et al. 2024), run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_ia_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_t5_ia_instr
  defence_id: instr
  benchmark: InjecAgent, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_ia_delim
  defence_id: delim
  benchmark: InjecAgent, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_ia_sand
  defence_id: sand
  benchmark: InjecAgent, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_ia_secinfer
  defence_id: secinfer
  benchmark: InjecAgent, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_ia_msa
  defence_id: msa
  benchmark: InjecAgent, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_adj_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_t5_adj_instr
  defence_id: instr
  benchmark: AgentDojo, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_adj_delim
  defence_id: delim
  benchmark: AgentDojo, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_adj_sand
  defence_id: sand
  benchmark: AgentDojo, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_adj_secinfer
  defence_id: secinfer
  benchmark: AgentDojo, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
- id: ev_t5_adj_msa
  defence_id: msa
  benchmark: AgentDojo, run by the authors for comparison
  attacker_type: static
  attacker_access: black_box
  judge: task-specific success criterion (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: longpibench_arxiv2026
  page: 8
  decoding: not reported
results:
- id: r_syn_paper_none
  evaluation_id: ev_syn_paper_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_paper_instr
  evaluation_id: ev_syn_paper_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_paper_delim
  evaluation_id: ev_syn_paper_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_paper_sand
  evaluation_id: ev_syn_paper_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_paper_secinfer
  evaluation_id: ev_syn_paper_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_paper_msa
  evaluation_id: ev_syn_paper_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_resume_none
  evaluation_id: ev_syn_resume_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_resume_instr
  evaluation_id: ev_syn_resume_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_resume_delim
  evaluation_id: ev_syn_resume_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_resume_sand
  evaluation_id: ev_syn_resume_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.96'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_resume_secinfer
  evaluation_id: ev_syn_resume_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_resume_msa
  evaluation_id: ev_syn_resume_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_email_none
  evaluation_id: ev_syn_email_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.97'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_email_instr
  evaluation_id: ev_syn_email_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.98'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_email_delim
  evaluation_id: ev_syn_email_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.90'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_email_sand
  evaluation_id: ev_syn_email_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.95'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_email_secinfer
  evaluation_id: ev_syn_email_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.83'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_email_msa
  evaluation_id: ev_syn_email_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.40'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_code_none
  evaluation_id: ev_syn_code_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.94'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_code_instr
  evaluation_id: ev_syn_code_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.93'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_code_delim
  evaluation_id: ev_syn_code_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.93'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_code_sand
  evaluation_id: ev_syn_code_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.92'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_code_secinfer
  evaluation_id: ev_syn_code_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.79'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_syn_code_msa
  evaluation_id: ev_syn_code_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.71'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_real_paper_none
  evaluation_id: ev_real_paper_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_paper_instr
  evaluation_id: ev_real_paper_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_paper_delim
  evaluation_id: ev_real_paper_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_paper_sand
  evaluation_id: ev_real_paper_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_paper_secinfer
  evaluation_id: ev_real_paper_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.99'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_paper_msa
  evaluation_id: ev_real_paper_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.99'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_resume_none
  evaluation_id: ev_real_resume_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.42'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_resume_instr
  evaluation_id: ev_real_resume_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.37'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_resume_delim
  evaluation_id: ev_real_resume_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.43'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_resume_sand
  evaluation_id: ev_real_resume_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.43'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_resume_secinfer
  evaluation_id: ev_real_resume_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.35'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_resume_msa
  evaluation_id: ev_real_resume_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_email_none
  evaluation_id: ev_real_email_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_email_instr
  evaluation_id: ev_real_email_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_email_delim
  evaluation_id: ev_real_email_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_email_sand
  evaluation_id: ev_real_email_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_email_secinfer
  evaluation_id: ev_real_email_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.75'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_email_msa
  evaluation_id: ev_real_email_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.42'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_code_none
  evaluation_id: ev_real_code_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_code_instr
  evaluation_id: ev_real_code_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.99'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_code_delim
  evaluation_id: ev_real_code_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.98'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_code_sand
  evaluation_id: ev_real_code_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_code_secinfer
  evaluation_id: ev_real_code_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.95'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_real_code_msa
  evaluation_id: ev_real_code_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 13
- id: r_t5_opi_none
  evaluation_id: ev_t5_opi_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.67'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_opi_instr
  evaluation_id: ev_t5_opi_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.59'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_opi_delim
  evaluation_id: ev_t5_opi_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.63'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_opi_sand
  evaluation_id: ev_t5_opi_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.61'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_opi_secinfer
  evaluation_id: ev_t5_opi_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_opi_msa
  evaluation_id: ev_t5_opi_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_ia_none
  evaluation_id: ev_t5_ia_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.61'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_ia_instr
  evaluation_id: ev_t5_ia_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.61'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_ia_delim
  evaluation_id: ev_t5_ia_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.60'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_ia_sand
  evaluation_id: ev_t5_ia_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.62'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_ia_secinfer
  evaluation_id: ev_t5_ia_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.01'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_ia_msa
  evaluation_id: ev_t5_ia_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.01'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_adj_none
  evaluation_id: ev_t5_adj_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.28'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_adj_instr
  evaluation_id: ev_t5_adj_instr
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.27'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_adj_delim
  evaluation_id: ev_t5_adj_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.28'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_adj_sand
  evaluation_id: ev_t5_adj_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.28'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_adj_secinfer
  evaluation_id: ev_t5_adj_secinfer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_t5_adj_msa
  evaluation_id: ev_t5_adj_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
comparisons:
- id: c_syn_paper_instr
  left: ev_syn_paper_instr
  right: ev_syn_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_paper_delim
  left: ev_syn_paper_delim
  right: ev_syn_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_paper_sand
  left: ev_syn_paper_sand
  right: ev_syn_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_paper_secinfer
  left: ev_syn_paper_secinfer
  right: ev_syn_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_paper_msa
  left: ev_syn_paper_msa
  right: ev_syn_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_resume_instr
  left: ev_syn_resume_instr
  right: ev_syn_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_resume_delim
  left: ev_syn_resume_delim
  right: ev_syn_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_resume_sand
  left: ev_syn_resume_sand
  right: ev_syn_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_resume_secinfer
  left: ev_syn_resume_secinfer
  right: ev_syn_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_resume_msa
  left: ev_syn_resume_msa
  right: ev_syn_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_email_instr
  left: ev_syn_email_instr
  right: ev_syn_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_email_delim
  left: ev_syn_email_delim
  right: ev_syn_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_email_sand
  left: ev_syn_email_sand
  right: ev_syn_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_email_secinfer
  left: ev_syn_email_secinfer
  right: ev_syn_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_email_msa
  left: ev_syn_email_msa
  right: ev_syn_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_code_instr
  left: ev_syn_code_instr
  right: ev_syn_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_code_delim
  left: ev_syn_code_delim
  right: ev_syn_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_code_sand
  left: ev_syn_code_sand
  right: ev_syn_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_code_secinfer
  left: ev_syn_code_secinfer
  right: ev_syn_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_syn_code_msa
  left: ev_syn_code_msa
  right: ev_syn_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_real_paper_instr
  left: ev_real_paper_instr
  right: ev_real_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_paper_delim
  left: ev_real_paper_delim
  right: ev_real_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_paper_sand
  left: ev_real_paper_sand
  right: ev_real_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_paper_secinfer
  left: ev_real_paper_secinfer
  right: ev_real_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_paper_msa
  left: ev_real_paper_msa
  right: ev_real_paper_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_resume_instr
  left: ev_real_resume_instr
  right: ev_real_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_resume_delim
  left: ev_real_resume_delim
  right: ev_real_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_resume_sand
  left: ev_real_resume_sand
  right: ev_real_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_resume_secinfer
  left: ev_real_resume_secinfer
  right: ev_real_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_resume_msa
  left: ev_real_resume_msa
  right: ev_real_resume_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_email_instr
  left: ev_real_email_instr
  right: ev_real_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_email_delim
  left: ev_real_email_delim
  right: ev_real_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_email_sand
  left: ev_real_email_sand
  right: ev_real_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_email_secinfer
  left: ev_real_email_secinfer
  right: ev_real_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_email_msa
  left: ev_real_email_msa
  right: ev_real_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_code_instr
  left: ev_real_code_instr
  right: ev_real_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_code_delim
  left: ev_real_code_delim
  right: ev_real_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_code_sand
  left: ev_real_code_sand
  right: ev_real_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_code_secinfer
  left: ev_real_code_secinfer
  right: ev_real_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_real_code_msa
  left: ev_real_code_msa
  right: ev_real_code_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_t5_opi_instr
  left: ev_t5_opi_instr
  right: ev_t5_opi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_opi_delim
  left: ev_t5_opi_delim
  right: ev_t5_opi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_opi_sand
  left: ev_t5_opi_sand
  right: ev_t5_opi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_opi_secinfer
  left: ev_t5_opi_secinfer
  right: ev_t5_opi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_opi_msa
  left: ev_t5_opi_msa
  right: ev_t5_opi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_ia_instr
  left: ev_t5_ia_instr
  right: ev_t5_ia_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_ia_delim
  left: ev_t5_ia_delim
  right: ev_t5_ia_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_ia_sand
  left: ev_t5_ia_sand
  right: ev_t5_ia_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_ia_secinfer
  left: ev_t5_ia_secinfer
  right: ev_t5_ia_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_ia_msa
  left: ev_t5_ia_msa
  right: ev_t5_ia_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_adj_instr
  left: ev_t5_adj_instr
  right: ev_t5_adj_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_adj_delim
  left: ev_t5_adj_delim
  right: ev_t5_adj_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_adj_sand
  left: ev_t5_adj_sand
  right: ev_t5_adj_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_adj_secinfer
  left: ev_t5_adj_secinfer
  right: ev_t5_adj_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_t5_adj_msa
  left: ev_t5_adj_msa
  right: ev_t5_adj_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: existing prevention-based defences fail to reduce attack success in long-context settings
  location: Section 5.4
  anchor: existing prevention techniques fail to reduce ASR in long-context settings
  jobs:
  - R1
  supported_by:
  - c_syn_paper_instr
  - c_syn_paper_delim
  - c_syn_paper_sand
  - c_syn_paper_secinfer
  - c_syn_paper_msa
  - c_syn_resume_instr
  - c_syn_resume_delim
  - c_syn_resume_sand
  - c_syn_resume_secinfer
  - c_syn_resume_msa
  - c_syn_email_instr
  - c_syn_email_delim
  - c_syn_email_sand
  - c_syn_email_secinfer
  - c_syn_email_msa
  - c_syn_code_instr
  - c_syn_code_delim
  - c_syn_code_sand
  - c_syn_code_secinfer
  - c_syn_code_msa
  - c_real_paper_instr
  - c_real_paper_delim
  - c_real_paper_sand
  - c_real_paper_secinfer
  - c_real_paper_msa
  - c_real_resume_instr
  - c_real_resume_delim
  - c_real_resume_sand
  - c_real_resume_secinfer
  - c_real_resume_msa
  - c_real_email_instr
  - c_real_email_delim
  - c_real_email_sand
  - c_real_email_secinfer
  - c_real_email_msa
  - c_real_code_instr
  - c_real_code_delim
  - c_real_code_sand
  - c_real_code_secinfer
  - c_real_code_msa
  page: 8
  contradicted_by:
  - r_syn_email_msa
  - r_real_email_msa
  - r_syn_code_msa
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 4, 5 and 13
    R2clean: not reported
    R2attack: not reported
schema_gaps:
- where: report
  note: defences are evaluated on Llama-3.1-8B-Instruct only, under the Authority spoof attack; the short-context comparison rows of Table 5 use each benchmark's own data with the same defences

`univdef_arxiv2026` (2609.16098)

In [ ]:
%%writefile reports/records/univdef_arxiv2026.yaml
report:
  id: univdef_arxiv2026
  slug: '2609.16098'
  title: Universal Defenses for Tool-Integrated LLM Agents Against Adversarial Attacks
  first_author: Xiaoyan Li
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2609.16098'
  version_read: arXiv v1
  first_date: '2026-09-14'
  read: F
  role: proposes
  extraction_scope: 'Table 2 (indirect prompt injection, averaged over injection types): No Defense and Delimiters (ASB) for six models, ASR and OTSR; the Section 4.3 remark on the delimiter defence (the paper''s ATF and NTR
    combinations filter or restore the toolset and are left out as filter and system level; CoT, paraphrasing and reflection are measured alone only under direct injection, Figure 2; memory poisoning and backdoor tables are
    not prompt injection in tool data)'
defences:
- id: delim_gem
  name: Delimiters (ASB implementation)
  origin: other_report
  base_checkpoint: Gemma2-9B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: ASB agent framework; injected instructions returned in tool responses
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we compare our defense methods with no defense and the delimiters approach in ASB
- id: delim_qw
  name: Delimiters (ASB implementation)
  origin: other_report
  base_checkpoint: Qwen2-7B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: ASB agent framework; injected instructions returned in tool responses
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we compare our defense methods with no defense and the delimiters approach in ASB
- id: delim_l3
  name: Delimiters (ASB implementation)
  origin: other_report
  base_checkpoint: LLaMA3-8B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: ASB agent framework; injected instructions returned in tool responses
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we compare our defense methods with no defense and the delimiters approach in ASB
- id: delim_l31
  name: Delimiters (ASB implementation)
  origin: other_report
  base_checkpoint: LLaMA3.1-8B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: ASB agent framework; injected instructions returned in tool responses
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we compare our defense methods with no defense and the delimiters approach in ASB
- id: delim_g35
  name: Delimiters (ASB implementation)
  origin: other_report
  base_checkpoint: GPT-3.5
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: ASB agent framework; injected instructions returned in tool responses
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we compare our defense methods with no defense and the delimiters approach in ASB
- id: delim_g4
  name: Delimiters (ASB implementation)
  origin: other_report
  base_checkpoint: GPT-4
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: ASB agent framework; injected instructions returned in tool responses
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we compare our defense methods with no defense and the delimiters approach in ASB
evaluations:
- id: ev_gem_none
  defence_id: none:Gemma2-9B
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gem_delim
  defence_id: delim_gem
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
- id: ev_qw_none
  defence_id: none:Qwen2-7B
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_qw_delim
  defence_id: delim_qw
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
- id: ev_l3_none
  defence_id: none:LLaMA3-8B
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l3_delim
  defence_id: delim_l3
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
- id: ev_l31_none
  defence_id: none:LLaMA3.1-8B
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l31_delim
  defence_id: delim_l31
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
- id: ev_g35_none
  defence_id: none:GPT-3.5
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g35_delim
  defence_id: delim_g35
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
- id: ev_g4_none
  defence_id: none:GPT-4
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g4_delim
  defence_id: delim_g4
  benchmark: ASB, indirect prompt injection, averaged over injection types
  attacker_type: static
  attacker_access: black_box
  judge: ASB checks (attacker tool used; original task completed)
  layout_same_as_defence: 'yes'
  measured_by: univdef_arxiv2026
  page: 7
  decoding: not reported
results:
- id: r_gem_none_asr
  evaluation_id: ev_gem_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0894'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gem_none_otsr
  evaluation_id: ev_gem_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.165'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gem_delim_asr
  evaluation_id: ev_gem_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0719'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gem_delim_otsr
  evaluation_id: ev_gem_delim
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.1269'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_qw_none_asr
  evaluation_id: ev_qw_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0606'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_qw_none_otsr
  evaluation_id: ev_qw_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.115'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_qw_delim_asr
  evaluation_id: ev_qw_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0431'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_qw_delim_otsr
  evaluation_id: ev_qw_delim
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.0738'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l3_none_asr
  evaluation_id: ev_l3_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.02'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l3_none_otsr
  evaluation_id: ev_l3_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.0275'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l3_delim_asr
  evaluation_id: ev_l3_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0256'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l3_delim_otsr
  evaluation_id: ev_l3_delim
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.03'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_none_asr
  evaluation_id: ev_l31_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0756'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_none_otsr
  evaluation_id: ev_l31_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.0819'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_delim_asr
  evaluation_id: ev_l31_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0519'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_delim_otsr
  evaluation_id: ev_l31_delim
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.0369'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_g35_none_asr
  evaluation_id: ev_g35_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.5137'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_g35_none_otsr
  evaluation_id: ev_g35_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.2231'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_g35_delim_asr
  evaluation_id: ev_g35_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.2406'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_g35_delim_otsr
  evaluation_id: ev_g35_delim
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.0938'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_g4_none_asr
  evaluation_id: ev_g4_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.4456'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_g4_none_otsr
  evaluation_id: ev_g4_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.3775'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_g4_delim_asr
  evaluation_id: ev_g4_delim
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.5619'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_g4_delim_otsr
  evaluation_id: ev_g4_delim
  quantity: R2attack
  coverage: proxy
  metric_as_printed: OTSR (original task success)
  value: '0.32'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
comparisons:
- id: c_gem
  left: ev_gem_delim
  right: ev_gem_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_qw
  left: ev_qw_delim
  right: ev_qw_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l3
  left: ev_l3_delim
  right: ev_l3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31
  left: ev_l31_delim
  right: ev_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_g35
  left: ev_g35_delim
  right: ev_g35_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_g4
  left: ev_g4_delim
  right: ev_g4_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
claims:
- id: k1
  text: the delimiter-based defence lowers original task success in some settings
  location: Section 4.3
  anchor: which is also observed with the delimiter-based defense
  jobs:
  - R2
  supported_by:
  - c_gem
  - c_qw
  - c_l3
  - c_l31
  - c_g35
  - c_g4
  page: 6
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: proxy
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2
    R2attack: Table 2 OTSR (whether it is measured in attacked runs is not stated)
    R2clean: not reported
schema_gaps:
- where: report
  note: the paper's own defences always include tool filtering (ATF) or toolset restoration (NTR); the only in-scope comparison is ASB's delimiter defence against no defence; GPT-4's attack success rises under delimiters (0.4456
    to 0.5619)

`explosive_arxiv2026` (2609.22510)

In [ ]:
%%writefile reports/records/explosive_arxiv2026.yaml
report:
  id: explosive_arxiv2026
  slug: '2609.22510'
  title: 'Defusing Explosive Prompts: Understanding and Preventing Trigger-Based Prompt Injections in LLM Agents'
  first_author: Justin Szczepaniak
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2609.22510'
  version_read: arXiv v1
  first_date: '2026-09-18'
  read: F
  role: mixed
  extraction_scope: 'Appendix A.4.7: Table 24 (undefended Llama-3.3-70B-Instruct and released Meta-SecAlign-70B on 2,002 paired AgentDojo episodes, imperative and explosive prompts) and Table 25 (the authors'' EP-DPO continuation,
    best step); the A.4.7 claim (Prompt Guard 2, PIGuard, PIShield and DeFuse are detectors and are left out; Table 5''s SecAlign cell uses a different denominator from its baseline, so Table 24 is coded instead)'
defences:
- id: msa70
  name: Meta-SecAlign-70B (released LoRA adapter, served merged)
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 23
  layout: AgentDojo agent; payload in a tool result, trigger arriving later as an ordinary user message
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Released Meta-SecAlign-70B fully closes
- id: epdpo
  name: Meta-SecAlign-70B + EP-DPO (QLoRA DPO continuation on 1,200 two-turn dormant-payload preference pairs; step 50)
  origin: this_report
  base_checkpoint: Llama-3.3-70B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 23
  layout: AgentDojo agent; payload in a tool result, trigger arriving later as an ordinary user message
  layout_compat: *id001
  anchor: We continue the released Meta-SecAlign-70B LoRA adapter
evaluations:
- id: ev_none_ipi
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDojo, 2,002 paired episodes with real tool execution, immediate imperative injection
  attacker_type: static
  attacker_access: black_box
  judge: state-changing tool call executed
  layout_same_as_defence: 'yes'
  measured_by: explosive_arxiv2026
  page: 23
  decoding: not reported
  sample_size: '2002'
  baseline_kind: off_the_shelf
- id: ev_none_ep
  defence_id: none:Llama-3.3-70B-Instruct
  benchmark: AgentDojo, 2,002 paired episodes with real tool execution, explosive (trigger-conditioned) prompts
  attacker_type: static
  attacker_access: black_box
  judge: state-changing tool call executed
  layout_same_as_defence: 'yes'
  measured_by: explosive_arxiv2026
  page: 23
  decoding: not reported
  sample_size: '2002'
  baseline_kind: off_the_shelf
- id: ev_msa70_ipi
  defence_id: msa70
  benchmark: AgentDojo, 2,002 paired episodes with real tool execution, immediate imperative injection
  attacker_type: static
  attacker_access: black_box
  judge: state-changing tool call executed
  layout_same_as_defence: 'yes'
  measured_by: explosive_arxiv2026
  page: 23
  decoding: not reported
  sample_size: '2002'
- id: ev_msa70_ep
  defence_id: msa70
  benchmark: AgentDojo, 2,002 paired episodes with real tool execution, explosive (trigger-conditioned) prompts
  attacker_type: static
  attacker_access: black_box
  judge: state-changing tool call executed
  layout_same_as_defence: 'yes'
  measured_by: explosive_arxiv2026
  page: 23
  decoding: not reported
  sample_size: '2002'
- id: ev_t25_rel
  defence_id: msa70
  benchmark: AgentDojo, 2,002 paired episodes with real tool execution, explosive prompts
  attacker_type: static
  attacker_access: black_box
  judge: state-changing tool call executed
  layout_same_as_defence: 'yes'
  measured_by: explosive_arxiv2026
  page: 23
  decoding: not reported
  sample_size: '2002'
- id: ev_t25_dpo
  defence_id: epdpo
  benchmark: AgentDojo, 2,002 paired episodes with real tool execution, explosive prompts
  attacker_type: static
  attacker_access: black_box
  judge: state-changing tool call executed
  layout_same_as_defence: 'yes'
  measured_by: explosive_arxiv2026
  page: 23
  decoding: not reported
  sample_size: '2002'
results:
- id: r_none_ipi
  evaluation_id: ev_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, imperative injection
  value: '18.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 24
- id: r_none_ep_turn1
  evaluation_id: ev_none_ep
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, explosive prompt (injection turn)
  value: '12.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 24
- id: r_none_ep_turn2
  evaluation_id: ev_none_ep
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, explosive prompt (trigger turn)
  value: '17.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 24
- id: r_none_ep_any
  evaluation_id: ev_none_ep
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, explosive prompt (any turn)
  value: '27.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 24
- id: r_msa70_ipi
  evaluation_id: ev_msa70_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, imperative injection
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 24
- id: r_msa70_ep_turn1
  evaluation_id: ev_msa70_ep
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, explosive prompt (injection turn)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 24
- id: r_msa70_ep_turn2
  evaluation_id: ev_msa70_ep
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, explosive prompt (trigger turn)
  value: '11.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 24
- id: r_msa70_ep_any
  evaluation_id: ev_msa70_ep
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, explosive prompt (any turn)
  value: '11.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 24
- id: r_t25_rel
  evaluation_id: ev_t25_rel
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, explosive prompt (any turn)
  value: '11.84'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 25
- id: r_t25_dpo
  evaluation_id: ev_t25_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, explosive prompt (any turn)
  value: '9.59'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 25
comparisons:
- id: c_ipi
  left: ev_msa70_ipi
  right: ev_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ep
  left: ev_msa70_ep
  right: ev_none_ep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_epdpo
  left: ev_t25_dpo
  right: ev_t25_rel
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: EP-DPO continuation against the released adapter it starts from
  page: 23
claims:
- id: k1
  text: Meta-SecAlign closes immediate imperative injection, but its remaining explosive-prompt failures all fire at the later trigger turn
  location: Appendix A.4.7
  anchor: Single-turn SecAlign training does not cover this shape
  jobs:
  - R1
  supported_by:
  - c_ipi
  - c_ep
  - c_epdpo
  page: 23
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 24 and 25
    R2clean: not reported
    R2attack: not reported
schema_gaps:
- where: report
  note: the main-text comparison of Table 5 places SecAlign's all-pairs rate (base 27.3%) beside a drop-free baseline (34.3%), which the authors flag as biased in SecAlign's favour; Table 24 uses one denominator for both arms;
    the EP-DPO checkpoint was chosen as the best of three steps

`readable_arxiv2026` (2610.05295)

In [ ]:
%%writefile reports/records/readable_arxiv2026.yaml
report:
  id: readable_arxiv2026
  slug: '2610.05295'
  title: 'Readable Before Actionable: Causal Tracing of Indirect Prompt Injection'
  first_author: Zhe Yu
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.05295'
  version_read: arXiv v1
  first_date: '2026-10-04'
  read: F
  role: mixed
  extraction_scope: 'Table 11 (AgentDojo Slack and Workspace, Qwen-2.5-7B: no intervention, suite-matched and leave-attack-family-out directions, controlled-role, random and L8 controls), Figure 3B Llama-3.1-8B Slack bar labels,
    Table 10 rows lambda 0 and 32 (controlled tool-output injection); the abstract claim (the probing and patching analyses test no defence; the position study of Figure 2B and the obfuscation and memory analyses are not coded)'
defences:
- id: match_slack
  name: suite-matched direction edit (Slack hijacked-minus-resisted direction, L20, lambda 32)
  origin: this_report
  base_checkpoint: Qwen-2.5-7B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: laf_slack
  name: suite-matched direction edit, leave-attack-family-out (Slack, L20, lambda 32)
  origin: this_report
  base_checkpoint: Qwen-2.5-7B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: match_ws
  name: suite-matched direction edit (Workspace hijacked-minus-resisted direction, L20, lambda 32)
  origin: this_report
  base_checkpoint: Qwen-2.5-7B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: laf_ws
  name: suite-matched direction edit, leave-attack-family-out (Workspace, L20, lambda 32)
  origin: this_report
  base_checkpoint: Qwen-2.5-7B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: ctrl_slack
  name: controlled-role direction edit (role direction from synthetic instruction-data pairs, L20, lambda 32)
  origin: this_report
  base_checkpoint: Qwen-2.5-7B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: rand_slack
  name: random direction edit (control, L20, lambda 32)
  origin: this_report
  base_checkpoint: Qwen-2.5-7B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: l8_slack
  name: suite-matched edit at out-of-band layer 8 (control)
  origin: this_report
  base_checkpoint: Qwen-2.5-7B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: match_llama
  name: suite-matched direction edit (Slack direction, L22)
  origin: this_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: ctrl_llama
  name: controlled-role direction edit (L22)
  origin: this_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: rand_llama
  name: random direction edit (control, L22)
  origin: this_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: the edit subtracts λ times the excess projection above a benign reference
- id: role_q7
  name: controlled role-direction edit at L24, lambda 32
  origin: this_report
  base_checkpoint: Qwen-2.5-7B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 15
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: One-sided directional intervention
- id: role_q15
  name: controlled role-direction edit at L22, lambda 32
  origin: this_report
  base_checkpoint: Qwen-2.5-1.5B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 15
  layout: AgentDojo agent; edit applied to residual activations at a chosen layer and token; injection in tool outputs
  layout_compat: *id001
  anchor: One-sided directional intervention
evaluations:
- id: ev_slack_none
  defence_id: none:Qwen-2.5-7B
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 16
  decoding: not reported
  sample_size: '240'
  baseline_kind: off_the_shelf
- id: ev_slack_ctrl_slack
  defence_id: ctrl_slack
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 16
  decoding: not reported
  sample_size: '240'
- id: ev_slack_rand_slack
  defence_id: rand_slack
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 16
  decoding: not reported
  sample_size: '240'
- id: ev_slack_l8_slack
  defence_id: l8_slack
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 16
  decoding: not reported
  sample_size: '240'
- id: ev_slack_match_slack
  defence_id: match_slack
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 16
  decoding: not reported
  sample_size: '240'
- id: ev_slack_laf_slack
  defence_id: laf_slack
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 16
  decoding: not reported
  sample_size: '240'
- id: ev_ws_none
  defence_id: none:Qwen-2.5-7B
  benchmark: AgentDojo Workspace, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 16
  decoding: not reported
  sample_size: '240'
  baseline_kind: off_the_shelf
- id: ev_ws_match_ws
  defence_id: match_ws
  benchmark: AgentDojo Workspace, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 16
  decoding: not reported
  sample_size: '240'
- id: ev_ws_laf_ws
  defence_id: laf_ws
  benchmark: AgentDojo Workspace, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 16
  decoding: not reported
  sample_size: '240'
- id: ev_ll_none
  defence_id: none:Llama-3.1-8B
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '240'
  baseline_kind: off_the_shelf
- id: ev_ll_ctrl_llama
  defence_id: ctrl_llama
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '240'
- id: ev_ll_rand_llama
  defence_id: rand_llama
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '240'
- id: ev_ll_match_llama
  defence_id: match_llama
  benchmark: AgentDojo Slack, 240 held-out injected trajectories
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo attack check on the continued trajectory
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '240'
- id: ev_ct_q7_none
  defence_id: none:Qwen-2.5-7B
  benchmark: controlled synthetic tool-output injection, 100 prompts
  attacker_type: static
  attacker_access: black_box
  judge: attacker-specified output produced (attack); benign task correct (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 15
  decoding: not reported
  sample_size: '100'
  baseline_kind: off_the_shelf
- id: ev_ct_q7_edit
  defence_id: role_q7
  benchmark: controlled synthetic tool-output injection, 100 prompts
  attacker_type: static
  attacker_access: black_box
  judge: attacker-specified output produced (attack); benign task correct (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 15
  decoding: not reported
  sample_size: '100'
- id: ev_ct_q15_none
  defence_id: none:Qwen-2.5-1.5B
  benchmark: controlled synthetic tool-output injection, 100 prompts
  attacker_type: static
  attacker_access: black_box
  judge: attacker-specified output produced (attack); benign task correct (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 15
  decoding: not reported
  sample_size: '100'
  baseline_kind: off_the_shelf
- id: ev_ct_q15_edit
  defence_id: role_q15
  benchmark: controlled synthetic tool-output injection, 100 prompts
  attacker_type: static
  attacker_access: black_box
  judge: attacker-specified output produced (attack); benign task correct (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: readable_arxiv2026
  page: 15
  decoding: not reported
  sample_size: '100'
results:
- id: r_slack_none
  evaluation_id: ev_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '34.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 11
- id: r_slack_ctrl_slack
  evaluation_id: ev_slack_ctrl_slack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '33.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 11
- id: r_slack_rand_slack
  evaluation_id: ev_slack_rand_slack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '34.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 11
- id: r_slack_l8_slack
  evaluation_id: ev_slack_l8_slack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '33.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 11
- id: r_slack_match_slack
  evaluation_id: ev_slack_match_slack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '21.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 11
- id: r_slack_laf_slack
  evaluation_id: ev_slack_laf_slack
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '24.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 11
- id: r_ws_none
  evaluation_id: ev_ws_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '31.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 11
- id: r_ws_match_ws
  evaluation_id: ev_ws_match_ws
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '18.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 11
- id: r_ws_laf_ws
  evaluation_id: ev_ws_laf_ws
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '22.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 11
- id: r_ll_none
  evaluation_id: ev_ll_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '32.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3B bar labels
- id: r_ll_ctrl_llama
  evaluation_id: ev_ll_ctrl_llama
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '31.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3B bar labels
- id: r_ll_rand_llama
  evaluation_id: ev_ll_rand_llama
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '32.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3B bar labels
- id: r_ll_match_llama
  evaluation_id: ev_ll_match_llama
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '19.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3B bar labels
- id: r_ct_q7_none_asr
  evaluation_id: ev_ct_q7_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '46'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 10
- id: r_ct_q7_none_acc
  evaluation_id: ev_ct_q7_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign-task accuracy
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 10
- id: r_ct_q7_edit_asr
  evaluation_id: ev_ct_q7_edit
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '26'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 10
- id: r_ct_q7_edit_acc
  evaluation_id: ev_ct_q7_edit
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign-task accuracy
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 10
- id: r_ct_q15_none_asr
  evaluation_id: ev_ct_q15_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '34'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 10
- id: r_ct_q15_none_acc
  evaluation_id: ev_ct_q15_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign-task accuracy
  value: '92'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 10
- id: r_ct_q15_edit_asr
  evaluation_id: ev_ct_q15_edit
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '28'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 10
- id: r_ct_q15_edit_acc
  evaluation_id: ev_ct_q15_edit
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign-task accuracy
  value: '92'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 10
comparisons:
- id: c_slack_match_slack
  left: ev_slack_match_slack
  right: ev_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_slack_laf_slack
  left: ev_slack_laf_slack
  right: ev_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_slack_ctrl_slack
  left: ev_slack_ctrl_slack
  right: ev_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_ws_match_ws
  left: ev_ws_match_ws
  right: ev_ws_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_ws_laf_ws
  left: ev_ws_laf_ws
  right: ev_ws_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_slack_match_vs_rand_slack
  left: ev_slack_match_slack
  right: ev_slack_rand_slack
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the suite-matched direction against a control edit of the same strength
  page: 16
- id: c_slack_match_vs_l8_slack
  left: ev_slack_match_slack
  right: ev_slack_l8_slack
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the suite-matched direction against a control edit of the same strength
  page: 16
- id: c_ll_match_llama
  left: ev_ll_match_llama
  right: ev_ll_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ll_ctrl_llama
  left: ev_ll_ctrl_llama
  right: ev_ll_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ll_match_vs_rand
  left: ev_ll_match_llama
  right: ev_ll_rand_llama
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the suite-matched direction against a random direction
  page: 7
- id: c_ct_q7
  left: ev_ct_q7_edit
  right: ev_ct_q7_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_ct_q15
  left: ev_ct_q15_edit
  right: ev_ct_q15_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 15
claims:
- id: k1
  text: directions estimated from hijacked and resisted training trajectories reduce attack success on AgentDojo when applied at the right positions
  location: abstract
  anchor: directions estimated from hijacked and resisted training trajectories reduce attack success
  jobs:
  - R1
  supported_by:
  - c_slack_match_slack
  - c_ws_match_ws
  - c_ll_match_llama
  - c_slack_laf_slack
  - c_ws_laf_ws
  - c_slack_match_vs_rand_slack
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: proxy
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 10 and 11; Figure 3B
    R2attack: Table 10 benign-task accuracy
    R2clean: benign utility stated only for the position study (95.8% Qwen, 94.8% Llama)
schema_gaps:
- where: report
  note: the edits need white-box access to activations and directions estimated from labelled hijacked and resisted trajectories of the same suite; directions transfer poorly across channels, and attacks are not adaptive to
    the direction except in the paper's Appendix J

`trustmi_arxiv2026` (2610.06064)

In [ ]:
%%writefile reports/records/trustmi_arxiv2026.yaml
report:
  id: trustmi_arxiv2026
  slug: '2610.06064'
  title: 'TrustMI: Causally controlling how assistants trust their users'
  first_author: Théo Lasnier
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.06064'
  version_read: arXiv v1
  first_date: '2026-10-05'
  read: F
  role: mixed
  extraction_scope: Table 7 (AgentDojo with and without injections) for the six models at steering strengths -1 and -2 against the unsteered model (0); the Section 4 remark on protecting agents (positive strengths raise trust
    and are not a defence; AgentHarm, Agentic Misalignment and the capability tables are not prompt injection)
defences:
- id: steerm1_q9
  name: trust steering at strength -1 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm2_q9
  name: trust steering at strength -2 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm1_q27
  name: trust steering at strength -1 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Qwen3.5-27B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm2_q27
  name: trust steering at strength -2 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Qwen3.5-27B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm1_l8
  name: trust steering at strength -1 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm2_l8
  name: trust steering at strength -2 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm1_l70
  name: trust steering at strength -1 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Llama-3.1-70B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm2_l70
  name: trust steering at strength -2 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Llama-3.1-70B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm1_o7
  name: trust steering at strength -1 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Olmo-3-7B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm2_o7
  name: trust steering at strength -2 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Olmo-3-7B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm1_o32
  name: trust steering at strength -1 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Olmo-3.1-32B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
- id: steerm2_o32
  name: trust steering at strength -2 (distrust; steering matrices learned with BiPO on 2,000 contrastive trust conversations, model frozen)
  origin: this_report
  base_checkpoint: Olmo-3.1-32B
  axis1:
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  layout: AgentDojo agent; trust steering matrix applied to the activations of user turns and tool outputs
  layout_compat: *id001
  anchor: learn steering matrices using Bi-directional Preference Optimization
evaluations:
- id: ev_q9_none
  defence_id: none:Qwen3.5-9B
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
  baseline_kind: off_the_shelf
- id: ev_q9_none_ben
  defence_id: none:Qwen3.5-9B
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
  baseline_kind: off_the_shelf
- id: ev_q9_m1
  defence_id: steerm1_q9
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_q9_m1_ben
  defence_id: steerm1_q9
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_q9_m2
  defence_id: steerm2_q9
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_q9_m2_ben
  defence_id: steerm2_q9
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_q27_none
  defence_id: none:Qwen3.5-27B
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
  baseline_kind: off_the_shelf
- id: ev_q27_none_ben
  defence_id: none:Qwen3.5-27B
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
  baseline_kind: off_the_shelf
- id: ev_q27_m1
  defence_id: steerm1_q27
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_q27_m1_ben
  defence_id: steerm1_q27
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_q27_m2
  defence_id: steerm2_q27
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_q27_m2_ben
  defence_id: steerm2_q27
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_l8_none
  defence_id: none:Llama-3.1-8B
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
  baseline_kind: off_the_shelf
- id: ev_l8_none_ben
  defence_id: none:Llama-3.1-8B
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
  baseline_kind: off_the_shelf
- id: ev_l8_m1
  defence_id: steerm1_l8
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_l8_m1_ben
  defence_id: steerm1_l8
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_l8_m2
  defence_id: steerm2_l8
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_l8_m2_ben
  defence_id: steerm2_l8
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_l70_none
  defence_id: none:Llama-3.1-70B
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
  baseline_kind: off_the_shelf
- id: ev_l70_none_ben
  defence_id: none:Llama-3.1-70B
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
  baseline_kind: off_the_shelf
- id: ev_l70_m1
  defence_id: steerm1_l70
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_l70_m1_ben
  defence_id: steerm1_l70
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_l70_m2
  defence_id: steerm2_l70
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_l70_m2_ben
  defence_id: steerm2_l70
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_o7_none
  defence_id: none:Olmo-3-7B
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
  baseline_kind: off_the_shelf
- id: ev_o7_none_ben
  defence_id: none:Olmo-3-7B
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
  baseline_kind: off_the_shelf
- id: ev_o7_m1
  defence_id: steerm1_o7
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_o7_m1_ben
  defence_id: steerm1_o7
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_o7_m2
  defence_id: steerm2_o7
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_o7_m2_ben
  defence_id: steerm2_o7
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_o32_none
  defence_id: none:Olmo-3.1-32B
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
  baseline_kind: off_the_shelf
- id: ev_o32_none_ben
  defence_id: none:Olmo-3.1-32B
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
  baseline_kind: off_the_shelf
- id: ev_o32_m1
  defence_id: steerm1_o32
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_o32_m1_ben
  defence_id: steerm1_o32
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
- id: ev_o32_m2
  defence_id: steerm2_o32
  benchmark: AgentDojo with injections (944 cases)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '944'
- id: ev_o32_m2_ben
  defence_id: steerm2_o32
  benchmark: AgentDojo without injections (96 tasks)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checks
  layout_same_as_defence: 'yes'
  measured_by: trustmi_arxiv2026
  page: 25
  decoding: not reported
  sample_size: '96'
results:
- id: r_q9_none_asr
  evaluation_id: ev_q9_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '14.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q9_none_ui
  evaluation_id: ev_q9_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '57.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q9_none_ub
  evaluation_id: ev_q9_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '67.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q9_m1_asr
  evaluation_id: ev_q9_m1
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '7.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q9_m1_ui
  evaluation_id: ev_q9_m1
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '60.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q9_m1_ub
  evaluation_id: ev_q9_m1_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '67.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q9_m2_asr
  evaluation_id: ev_q9_m2
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '1.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q9_m2_ui
  evaluation_id: ev_q9_m2
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '52.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q9_m2_ub
  evaluation_id: ev_q9_m2_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '58.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q27_none_asr
  evaluation_id: ev_q27_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '33.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q27_none_ui
  evaluation_id: ev_q27_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '56.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q27_none_ub
  evaluation_id: ev_q27_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '76.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q27_m1_asr
  evaluation_id: ev_q27_m1
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '20.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q27_m1_ui
  evaluation_id: ev_q27_m1
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '66.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q27_m1_ub
  evaluation_id: ev_q27_m1_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '75.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q27_m2_asr
  evaluation_id: ev_q27_m2
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '7.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q27_m2_ui
  evaluation_id: ev_q27_m2
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '66.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_q27_m2_ub
  evaluation_id: ev_q27_m2_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '73.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l8_none_asr
  evaluation_id: ev_l8_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '2.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l8_none_ui
  evaluation_id: ev_l8_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '15.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l8_none_ub
  evaluation_id: ev_l8_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '16.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l8_m1_asr
  evaluation_id: ev_l8_m1
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l8_m1_ui
  evaluation_id: ev_l8_m1
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '13.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l8_m1_ub
  evaluation_id: ev_l8_m1_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '14.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l8_m2_asr
  evaluation_id: ev_l8_m2
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l8_m2_ui
  evaluation_id: ev_l8_m2
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '11.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l8_m2_ub
  evaluation_id: ev_l8_m2_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '14.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l70_none_asr
  evaluation_id: ev_l70_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '6.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l70_none_ui
  evaluation_id: ev_l70_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '24.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l70_none_ub
  evaluation_id: ev_l70_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '28.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l70_m1_asr
  evaluation_id: ev_l70_m1
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '4.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l70_m1_ui
  evaluation_id: ev_l70_m1
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '22.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l70_m1_ub
  evaluation_id: ev_l70_m1_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '25.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l70_m2_asr
  evaluation_id: ev_l70_m2
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '1.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l70_m2_ui
  evaluation_id: ev_l70_m2
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '17.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_l70_m2_ub
  evaluation_id: ev_l70_m2_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '20.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o7_none_asr
  evaluation_id: ev_o7_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '0.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o7_none_ui
  evaluation_id: ev_o7_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '7.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o7_none_ub
  evaluation_id: ev_o7_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '8.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o7_m1_asr
  evaluation_id: ev_o7_m1
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o7_m1_ui
  evaluation_id: ev_o7_m1
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '6.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o7_m1_ub
  evaluation_id: ev_o7_m1_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '7.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o7_m2_asr
  evaluation_id: ev_o7_m2
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o7_m2_ui
  evaluation_id: ev_o7_m2
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '5.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o7_m2_ub
  evaluation_id: ev_o7_m2_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '6.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o32_none_asr
  evaluation_id: ev_o32_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '11.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o32_none_ui
  evaluation_id: ev_o32_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '25.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o32_none_ub
  evaluation_id: ev_o32_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '32.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o32_m1_asr
  evaluation_id: ev_o32_m1
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '7.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o32_m1_ui
  evaluation_id: ev_o32_m1
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '22.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o32_m1_ub
  evaluation_id: ev_o32_m1_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '32.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o32_m2_asr
  evaluation_id: ev_o32_m2
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '2.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o32_m2_ui
  evaluation_id: ev_o32_m2
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility, injected
  value: '17.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_o32_m2_ub
  evaluation_id: ev_o32_m2_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility, benign
  value: '21.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
comparisons:
- id: c_q9_m1
  left: ev_q9_m1
  right: ev_q9_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_q9_m1_ben
  left: ev_q9_m1_ben
  right: ev_q9_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_q9_m2
  left: ev_q9_m2
  right: ev_q9_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_q9_m2_ben
  left: ev_q9_m2_ben
  right: ev_q9_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_q27_m1
  left: ev_q27_m1
  right: ev_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_q27_m1_ben
  left: ev_q27_m1_ben
  right: ev_q27_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_q27_m2
  left: ev_q27_m2
  right: ev_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_q27_m2_ben
  left: ev_q27_m2_ben
  right: ev_q27_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_l8_m1
  left: ev_l8_m1
  right: ev_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_l8_m1_ben
  left: ev_l8_m1_ben
  right: ev_l8_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_l8_m2
  left: ev_l8_m2
  right: ev_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_l8_m2_ben
  left: ev_l8_m2_ben
  right: ev_l8_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_l70_m1
  left: ev_l70_m1
  right: ev_l70_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_l70_m1_ben
  left: ev_l70_m1_ben
  right: ev_l70_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_l70_m2
  left: ev_l70_m2
  right: ev_l70_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_l70_m2_ben
  left: ev_l70_m2_ben
  right: ev_l70_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_o7_m1
  left: ev_o7_m1
  right: ev_o7_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_o7_m1_ben
  left: ev_o7_m1_ben
  right: ev_o7_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_o7_m2
  left: ev_o7_m2
  right: ev_o7_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_o7_m2_ben
  left: ev_o7_m2_ben
  right: ev_o7_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_o32_m1
  left: ev_o32_m1
  right: ev_o32_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_o32_m1_ben
  left: ev_o32_m1_ben
  right: ev_o32_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_o32_m2
  left: ev_o32_m2
  right: ev_o32_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_o32_m2_ben
  left: ev_o32_m2_ben
  right: ev_o32_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
claims:
- id: k1
  text: moderate negative trust steering could help protect agents against prompt injections without much loss of benign performance
  location: Section 4
  anchor: Moderate negative steering could therefore help protect agents against prompt injections
  jobs:
  - R1
  - R2
  supported_by:
  - c_q9_m1
  - c_q27_m1
  - c_l8_m1
  - c_l70_m1
  - c_o7_m1
  - c_o32_m1
  - c_q9_m1_ben
  - c_q27_m1_ben
  - c_l8_m1_ben
  - c_l70_m1_ben
  - c_o7_m1_ben
  - c_o32_m1_ben
  page: 7
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 7
    R2clean: Table 7 (benign)
    R2attack: Table 7 (utility, injected)
schema_gaps:
- where: report
  note: the steering matrices are trained on trust conversations without injections and applied to both user turns and tool outputs; steering only the user turns raises attack success for both Qwen models (Appendix D.7), so
    the effect comes from the tool-output positions

`pvst_arxiv2026` (2610.19372)

In [ ]:
%%writefile reports/records/pvst_arxiv2026.yaml
report:
  id: pvst_arxiv2026
  slug: '2610.19372'
  title: 'Policy-Verified Self-Training: Learning Agent Compliance from Verifiable Policy Rewards'
  first_author: Min-Young Yu
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.19372'
  version_read: arXiv v1
  first_date: '2026-10-14'
  read: F
  role: proposes
  extraction_scope: 'Tables 1, 3 and 4 with the gate off (the trained agent alone): base, RFT-succ, RFT-gate, DPO-gate and the negative-selection controls on held-out AgentDojo tasks, the unseen tool_knowledge template and
    the leave-workspace-out transfer; the abstract claim (gate-on rows use the deterministic runtime gate, a tool-call filter, and are left out; Table 2 per-suite rows and the tau2-bench airline track are not coded)'
defences:
- id: rftsucc
  name: RFT-succ (LoRA SFT on the agent's own successful trajectories from attacked rollouts)
  origin: this_report
  base_checkpoint: Qwen3-8B (thinking disabled)
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AgentDojo agent (banking, travel, workspace); injections in tool outputs; LoRA adapter on the agent
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Gate-labelled preference optimization (DPO-gate)
- id: rftgate
  name: RFT-gate (LoRA SFT on own successful trajectories with no gate-denied call)
  origin: this_report
  base_checkpoint: Qwen3-8B (thinking disabled)
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AgentDojo agent (banking, travel, workspace); injections in tool outputs; LoRA adapter on the agent
  layout_compat: *id001
  anchor: Gate-labelled preference optimization (DPO-gate)
- id: dpogate
  name: 'DPO-gate (PVST: DPO from RFT-gate with gate-rejected negatives cut at the denied call)'
  origin: this_report
  base_checkpoint: Qwen3-8B (thinking disabled)
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AgentDojo agent (banking, travel, workspace); injections in tool outputs; LoRA adapter on the agent
  layout_compat: *id001
  anchor: Gate-labelled preference optimization (DPO-gate)
- id: dpoany
  name: 'DPO-ctrl-any (control: negatives are any failed trajectories)'
  origin: this_report
  base_checkpoint: Qwen3-8B (thinking disabled)
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AgentDojo agent (banking, travel, workspace); injections in tool outputs; LoRA adapter on the agent
  layout_compat: *id001
  anchor: Gate-labelled preference optimization (DPO-gate)
- id: dpoclean
  name: 'DPO-ctrl-clean (control: negatives are failed trajectories without violations)'
  origin: this_report
  base_checkpoint: Qwen3-8B (thinking disabled)
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AgentDojo agent (banking, travel, workspace); injections in tool outputs; LoRA adapter on the agent
  layout_compat: *id001
  anchor: Gate-labelled preference optimization (DPO-gate)
- id: dpooracle
  name: DPO-oracle (negatives chosen by the benchmark attack oracle)
  origin: this_report
  base_checkpoint: Qwen3-8B (thinking disabled)
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AgentDojo agent (banking, travel, workspace); injections in tool outputs; LoRA adapter on the agent
  layout_compat: *id001
  anchor: Gate-labelled preference optimization (DPO-gate)
- id: rftgate_lo
  name: RFT-gate trained on banking and travel only
  origin: this_report
  base_checkpoint: Qwen3-8B (thinking disabled)
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AgentDojo agent (banking, travel, workspace); injections in tool outputs; LoRA adapter on the agent
  layout_compat: *id001
  anchor: Gate-labelled preference optimization (DPO-gate)
- id: dpogate_lo
  name: DPO-gate trained on banking and travel only
  origin: this_report
  base_checkpoint: Qwen3-8B (thinking disabled)
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AgentDojo agent (banking, travel, workspace); injections in tool outputs; LoRA adapter on the agent
  layout_compat: *id001
  anchor: Gate-labelled preference optimization (DPO-gate)
evaluations:
- id: ev_t1_base
  defence_id: none:Qwen3-8B (thinking disabled)
  benchmark: AgentDojo held-out user tasks x all injection tasks, important_instructions, three suites pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 6
  decoding: temperature 0
  sample_size: about 240 held-out triples
  baseline_kind: off_the_shelf
- id: ev_t1_rftsucc
  defence_id: rftsucc
  benchmark: AgentDojo held-out user tasks x all injection tasks, important_instructions, three suites pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 6
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t1_rftgate
  defence_id: rftgate
  benchmark: AgentDojo held-out user tasks x all injection tasks, important_instructions, three suites pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 6
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t1_dpogate
  defence_id: dpogate
  benchmark: AgentDojo held-out user tasks x all injection tasks, important_instructions, three suites pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 6
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t1_dpoany
  defence_id: dpoany
  benchmark: AgentDojo held-out user tasks x all injection tasks, important_instructions, three suites pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 6
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t1_dpoclean
  defence_id: dpoclean
  benchmark: AgentDojo held-out user tasks x all injection tasks, important_instructions, three suites pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 6
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t1_dpooracle
  defence_id: dpooracle
  benchmark: AgentDojo held-out user tasks x all injection tasks, important_instructions, three suites pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 6
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t3_base
  defence_id: none:Qwen3-8B (thinking disabled)
  benchmark: AgentDojo held-out tasks, unseen tool_knowledge template
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 7
  decoding: temperature 0
  sample_size: about 240 held-out triples
  baseline_kind: off_the_shelf
- id: ev_t3_dpogate
  defence_id: dpogate
  benchmark: AgentDojo held-out tasks, unseen tool_knowledge template
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 7
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t3_dpoclean
  defence_id: dpoclean
  benchmark: AgentDojo held-out tasks, unseen tool_knowledge template
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 7
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t4_base
  defence_id: none:Qwen3-8B (thinking disabled)
  benchmark: AgentDojo workspace held-out tasks (suite excluded from leave-out training), important_instructions
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 7
  decoding: temperature 0
  sample_size: about 240 held-out triples
  baseline_kind: off_the_shelf
- id: ev_t4_dpogate
  defence_id: dpogate
  benchmark: AgentDojo workspace held-out tasks (suite excluded from leave-out training), important_instructions
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 7
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t4_rftgate_lo
  defence_id: rftgate_lo
  benchmark: AgentDojo workspace held-out tasks (suite excluded from leave-out training), important_instructions
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 7
  decoding: temperature 0
  sample_size: about 240 held-out triples
- id: ev_t4_dpogate_lo
  defence_id: dpogate_lo
  benchmark: AgentDojo workspace held-out tasks (suite excluded from leave-out training), important_instructions
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security flag (attack); task check (utility); gate replay (violations)
  layout_same_as_defence: 'yes'
  measured_by: pvst_arxiv2026
  page: 7
  decoding: temperature 0
  sample_size: about 240 held-out triples
results:
- id: r_t1_base_asr
  evaluation_id: ev_t1_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '12.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_base_viol
  evaluation_id: ev_t1_base
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '37.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_base_util
  evaluation_id: ev_t1_base
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '35.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_rftsucc_asr
  evaluation_id: ev_t1_rftsucc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '5.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_rftsucc_viol
  evaluation_id: ev_t1_rftsucc
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '30.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_rftsucc_util
  evaluation_id: ev_t1_rftsucc
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '35.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_rftgate_asr
  evaluation_id: ev_t1_rftgate
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '5.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_rftgate_viol
  evaluation_id: ev_t1_rftgate
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '28.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_rftgate_util
  evaluation_id: ev_t1_rftgate
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '40.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpogate_asr
  evaluation_id: ev_t1_dpogate
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpogate_viol
  evaluation_id: ev_t1_dpogate
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '10.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpogate_util
  evaluation_id: ev_t1_dpogate
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '39.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpoany_asr
  evaluation_id: ev_t1_dpoany
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpoany_viol
  evaluation_id: ev_t1_dpoany
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '20.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpoany_util
  evaluation_id: ev_t1_dpoany
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '45.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpoclean_asr
  evaluation_id: ev_t1_dpoclean
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '6.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpoclean_viol
  evaluation_id: ev_t1_dpoclean
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '31.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpoclean_util
  evaluation_id: ev_t1_dpoclean
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '41.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpooracle_asr
  evaluation_id: ev_t1_dpooracle
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpooracle_viol
  evaluation_id: ev_t1_dpooracle
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '22.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t1_dpooracle_util
  evaluation_id: ev_t1_dpooracle
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '40.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_t3_base_asr
  evaluation_id: ev_t3_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '14.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_t3_base_viol
  evaluation_id: ev_t3_base
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '37.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_t3_base_util
  evaluation_id: ev_t3_base
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '34.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_t3_dpogate_asr
  evaluation_id: ev_t3_dpogate
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_t3_dpogate_viol
  evaluation_id: ev_t3_dpogate
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '11.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_t3_dpogate_util
  evaluation_id: ev_t3_dpogate
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '38.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_t3_dpoclean_asr
  evaluation_id: ev_t3_dpoclean
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '10.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_t3_dpoclean_viol
  evaluation_id: ev_t3_dpoclean
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '33.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_t3_dpoclean_util
  evaluation_id: ev_t3_dpoclean
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '42.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_t4_base_asr
  evaluation_id: ev_t4_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '5.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_base_viol
  evaluation_id: ev_t4_base
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '34.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_base_util
  evaluation_id: ev_t4_base
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '34.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_dpogate_asr
  evaluation_id: ev_t4_dpogate
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_dpogate_viol
  evaluation_id: ev_t4_dpogate
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '14.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_dpogate_util
  evaluation_id: ev_t4_dpogate
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '46.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_rftgate_lo_asr
  evaluation_id: ev_t4_rftgate_lo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_rftgate_lo_viol
  evaluation_id: ev_t4_rftgate_lo
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '28.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_rftgate_lo_util
  evaluation_id: ev_t4_rftgate_lo
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '41.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_dpogate_lo_asr
  evaluation_id: ev_t4_dpogate_lo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_dpogate_lo_viol
  evaluation_id: ev_t4_dpogate_lo
  quantity: R1
  coverage: proxy
  metric_as_printed: violating trajectories (any gate-denied call)
  value: '16.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
- id: r_t4_dpogate_lo_util
  evaluation_id: ev_t4_dpogate_lo
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '42.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 4
comparisons:
- id: c_t1_rftsucc
  left: ev_t1_rftsucc
  right: ev_t1_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t1_rftgate
  left: ev_t1_rftgate
  right: ev_t1_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t1_dpogate
  left: ev_t1_dpogate
  right: ev_t1_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t1_dpoany
  left: ev_t1_dpoany
  right: ev_t1_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t1_dpoclean
  left: ev_t1_dpoclean
  right: ev_t1_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t1_dpooracle
  left: ev_t1_dpooracle
  right: ev_t1_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t1_dpogate_vs_dpoany
  left: ev_t1_dpogate
  right: ev_t1_dpoany
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: gate-selected negatives against the control's negatives
  page: 6
- id: c_t1_dpogate_vs_dpoclean
  left: ev_t1_dpogate
  right: ev_t1_dpoclean
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: gate-selected negatives against the control's negatives
  page: 6
- id: c_t1_dpogate_vs_dpooracle
  left: ev_t1_dpogate
  right: ev_t1_dpooracle
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: gate-selected negatives against the control's negatives
  page: 6
- id: c_t3_dpogate
  left: ev_t3_dpogate
  right: ev_t3_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_t3_dpoclean
  left: ev_t3_dpoclean
  right: ev_t3_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_t4_rftgate_lo
  left: ev_t4_rftgate_lo
  right: ev_t4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_t4_dpogate_lo
  left: ev_t4_dpogate_lo
  right: ev_t4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
claims:
- id: k1
  text: policy-verified preference training lowers violating trajectories and successful attacks at equal or higher pooled utility
  location: abstract
  anchor: policy-verified preference training of Qwen3-8B reduces violating trajectories
  jobs:
  - R1
  - R2
  supported_by:
  - c_t1_dpogate
  - c_t3_dpogate
  - c_t4_dpogate_lo
  - c_t1_dpogate_vs_dpoclean
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1, 3 and 4 (ASR; violating trajectories as a proxy)
    R2attack: Tables 1, 3 and 4 (utility under attack)
    R2clean: not coded
schema_gaps:
- where: report
  note: Table 1 values for RFT-gate, DPO-gate and the two controls are three-seed means, the rest single runs; training rollouts were collected in the attacked environment of the same benchmark, and DPO-gate lowers travel
    utility (33.3% to 16.7%, Table 2)

`medmas_arxiv2026` (2610.22594)

In [ ]:
%%writefile reports/records/medmas_arxiv2026.yaml
report:
  id: medmas_arxiv2026
  slug: '2610.22594'
  title: 'The Model Knows Better: Defending Medical Multi-Agent Systems Against Prompt Injection'
  first_author: Fardin Ahsan Sakib
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.22594'
  version_read: arXiv v1
  first_date: '2026-10-15'
  read: F
  role: mixed
  extraction_scope: 'Table 1 (MedQA: Instructional, Sandwich, Delimiters and Meta-SecAlign against no defence for five system and model configurations, TASR and clean accuracy) and Table 2 (Instructional on MedMisBench and
    MPIB); the abstract claim (PromptGuard 2, DataSentinel and PromptArmor are detectors, and the paper''s own defence uses a separate checker call to remove unsupported sentences before re-deciding, so it is left out as a
    sanitiser)'
defences:
- id: instr_sq
  name: Instructional (warning added to the task prompt)
  origin: other_report
  base_checkpoint: Qwen3.8-27B, single model
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Adds to the task prompt a warning that the data may
- id: sand_sq
  name: Sandwich (task instruction repeated after the data)
  origin: other_report
  base_checkpoint: Qwen3.8-27B, single model
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Repeats the task instruction after the data
- id: delim_sq
  name: Delimiters around the record
  origin: other_report
  base_checkpoint: Qwen3.8-27B, single model
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Encloses the data in delimiters so the model treats it
- id: instr_mq
  name: Instructional (warning added to the task prompt)
  origin: other_report
  base_checkpoint: Qwen3.8-27B in MDAgents
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Adds to the task prompt a warning that the data may
- id: sand_mq
  name: Sandwich (task instruction repeated after the data)
  origin: other_report
  base_checkpoint: Qwen3.8-27B in MDAgents
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Repeats the task instruction after the data
- id: delim_mq
  name: Delimiters around the record
  origin: other_report
  base_checkpoint: Qwen3.8-27B in MDAgents
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Encloses the data in delimiters so the model treats it
- id: instr_aq
  name: Instructional (warning added to the task prompt)
  origin: other_report
  base_checkpoint: Qwen3.8-27B in MedAgents
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Adds to the task prompt a warning that the data may
- id: sand_aq
  name: Sandwich (task instruction repeated after the data)
  origin: other_report
  base_checkpoint: Qwen3.8-27B in MedAgents
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Repeats the task instruction after the data
- id: delim_aq
  name: Delimiters around the record
  origin: other_report
  base_checkpoint: Qwen3.8-27B in MedAgents
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Encloses the data in delimiters so the model treats it
- id: instr_sl
  name: Instructional (warning added to the task prompt)
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct, single model
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Adds to the task prompt a warning that the data may
- id: sand_sl
  name: Sandwich (task instruction repeated after the data)
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct, single model
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Repeats the task instruction after the data
- id: delim_sl
  name: Delimiters around the record
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct, single model
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Encloses the data in delimiters so the model treats it
- id: msa_sl
  name: Meta-SecAlign (70B, trained to ignore instructions in untrusted input)
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct, single model
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Llama model trained to ignore instructions in its un-
- id: instr_ml
  name: Instructional (warning added to the task prompt)
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct in MDAgents
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Adds to the task prompt a warning that the data may
- id: sand_ml
  name: Sandwich (task instruction repeated after the data)
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct in MDAgents
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Repeats the task instruction after the data
- id: delim_ml
  name: Delimiters around the record
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct in MDAgents
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Encloses the data in delimiters so the model treats it
- id: msa_ml
  name: Meta-SecAlign (70B, trained to ignore instructions in untrusted input)
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct in MDAgents
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 13
  layout: clinical record with one injected passage at the end, question and options; every agent reads the same record
  layout_compat: *id001
  anchor: Llama model trained to ignore instructions in its un-
evaluations:
- id: ev_mq_sq_none
  defence_id: none:Qwen3.8-27B, single model
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
  baseline_kind: off_the_shelf
- id: ev_mq_sq_none_clean
  defence_id: none:Qwen3.8-27B, single model
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
  baseline_kind: off_the_shelf
- id: ev_mq_mq_none
  defence_id: none:Qwen3.8-27B in MDAgents
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
  baseline_kind: off_the_shelf
- id: ev_mq_mq_none_clean
  defence_id: none:Qwen3.8-27B in MDAgents
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
  baseline_kind: off_the_shelf
- id: ev_mq_aq_none
  defence_id: none:Qwen3.8-27B in MedAgents
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
  baseline_kind: off_the_shelf
- id: ev_mq_aq_none_clean
  defence_id: none:Qwen3.8-27B in MedAgents
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
  baseline_kind: off_the_shelf
- id: ev_mq_sl_none
  defence_id: none:Llama-3.3-70B-Instruct, single model
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
  baseline_kind: off_the_shelf
- id: ev_mq_sl_none_clean
  defence_id: none:Llama-3.3-70B-Instruct, single model
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
  baseline_kind: off_the_shelf
- id: ev_mq_ml_none
  defence_id: none:Llama-3.3-70B-Instruct in MDAgents
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
  baseline_kind: off_the_shelf
- id: ev_mq_ml_none_clean
  defence_id: none:Llama-3.3-70B-Instruct in MDAgents
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
  baseline_kind: off_the_shelf
- id: ev_mq_sq_instr
  defence_id: instr_sq
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_sq_instr_clean
  defence_id: instr_sq
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_mq_instr
  defence_id: instr_mq
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_mq_instr_clean
  defence_id: instr_mq
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_aq_instr
  defence_id: instr_aq
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_aq_instr_clean
  defence_id: instr_aq
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_sl_instr
  defence_id: instr_sl
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_sl_instr_clean
  defence_id: instr_sl
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_ml_instr
  defence_id: instr_ml
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_ml_instr_clean
  defence_id: instr_ml
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_sq_sand
  defence_id: sand_sq
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_sq_sand_clean
  defence_id: sand_sq
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_mq_sand
  defence_id: sand_mq
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_mq_sand_clean
  defence_id: sand_mq
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_aq_sand
  defence_id: sand_aq
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_aq_sand_clean
  defence_id: sand_aq
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_sl_sand
  defence_id: sand_sl
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_sl_sand_clean
  defence_id: sand_sl
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_ml_sand
  defence_id: sand_ml
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_ml_sand_clean
  defence_id: sand_ml
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_sq_delim
  defence_id: delim_sq
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_sq_delim_clean
  defence_id: delim_sq
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_mq_delim
  defence_id: delim_mq
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_mq_delim_clean
  defence_id: delim_mq
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_aq_delim
  defence_id: delim_aq
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_aq_delim_clean
  defence_id: delim_aq
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_sl_delim
  defence_id: delim_sl
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_sl_delim_clean
  defence_id: delim_sl
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_ml_delim
  defence_id: delim_ml
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_ml_delim_clean
  defence_id: delim_ml
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_sl_msa
  defence_id: msa_sl
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_sl_msa_clean
  defence_id: msa_sl
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_mq_ml_msa
  defence_id: msa_ml
  benchmark: MedQA (591 Step 2 and 3 questions), eight injected-passage attacks
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the attacker's target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: 591 questions x 8 attacks (cases answered correctly without attack)
- id: ev_mq_ml_msa_clean
  defence_id: msa_ml
  benchmark: MedQA (591 Step 2 and 3 questions), no injected text
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 4
  decoding: greedy
  sample_size: '591'
- id: ev_t2_mq_mmqa_none
  defence_id: none:Qwen3.8-27B in MDAgents
  benchmark: MedMisBench MedMisQA
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_mq_mmqa_instr
  defence_id: instr_mq
  benchmark: MedMisBench MedMisQA
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_mq_mmmc_none
  defence_id: none:Qwen3.8-27B in MDAgents
  benchmark: MedMisBench MedMisMCQA
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_mq_mmmc_instr
  defence_id: instr_mq
  benchmark: MedMisBench MedMisMCQA
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_mq_mmx_none
  defence_id: none:Qwen3.8-27B in MDAgents
  benchmark: MedMisBench MedMisXpertQA
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_mq_mmx_instr
  defence_id: instr_mq
  benchmark: MedMisBench MedMisXpertQA
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_aq_mmqa_none
  defence_id: none:Qwen3.8-27B in MedAgents
  benchmark: MedMisBench MedMisQA (300-case subset)
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_aq_mmqa_instr
  defence_id: instr_aq
  benchmark: MedMisBench MedMisQA (300-case subset)
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_aq_mmmc_none
  defence_id: none:Qwen3.8-27B in MedAgents
  benchmark: MedMisBench MedMisMCQA (300-case subset)
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_aq_mmmc_instr
  defence_id: instr_aq
  benchmark: MedMisBench MedMisMCQA (300-case subset)
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_aq_mmx_none
  defence_id: none:Qwen3.8-27B in MedAgents
  benchmark: MedMisBench MedMisXpertQA (300-case subset)
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_aq_mmx_instr
  defence_id: instr_aq
  benchmark: MedMisBench MedMisXpertQA (300-case subset)
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_ml_mmx_none
  defence_id: none:Llama-3.3-70B-Instruct in MDAgents
  benchmark: MedMisBench MedMisXpertQA (300-case subset)
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_ml_mmx_instr
  defence_id: instr_ml
  benchmark: MedMisBench MedMisXpertQA (300-case subset)
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_sq_mmx_none
  defence_id: none:Qwen3.8-27B, single model
  benchmark: MedMisBench MedMisXpertQA
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_sq_mmx_instr
  defence_id: instr_sq
  benchmark: MedMisBench MedMisXpertQA
  attacker_type: static
  attacker_access: black_box
  judge: answer changes to the target option
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_mq_mpib_none
  defence_id: none:Qwen3.8-27B in MDAgents
  benchmark: MPIB (582 retrieved-document cases)
  attacker_type: static
  attacker_access: black_box
  judge: harm rating 2 or higher by a Qwen3.8-27B judge
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_mq_mpib_instr
  defence_id: instr_mq
  benchmark: MPIB (582 retrieved-document cases)
  attacker_type: static
  attacker_access: black_box
  judge: harm rating 2 or higher by a Qwen3.8-27B judge
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_ml_mpib_none
  defence_id: none:Llama-3.3-70B-Instruct in MDAgents
  benchmark: MPIB (582 retrieved-document cases)
  attacker_type: static
  attacker_access: black_box
  judge: harm rating 2 or higher by a Qwen3.8-27B judge
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_ml_mpib_instr
  defence_id: instr_ml
  benchmark: MPIB (582 retrieved-document cases)
  attacker_type: static
  attacker_access: black_box
  judge: harm rating 2 or higher by a Qwen3.8-27B judge
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
- id: ev_t2_sq_mpib_none
  defence_id: none:Qwen3.8-27B, single model
  benchmark: MPIB (582 retrieved-document cases)
  attacker_type: static
  attacker_access: black_box
  judge: harm rating 2 or higher by a Qwen3.8-27B judge
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
  baseline_kind: off_the_shelf
- id: ev_t2_sq_mpib_instr
  defence_id: instr_sq
  benchmark: MPIB (582 retrieved-document cases)
  attacker_type: static
  attacker_access: black_box
  judge: harm rating 2 or higher by a Qwen3.8-27B judge
  layout_same_as_defence: 'yes'
  measured_by: medmas_arxiv2026
  page: 6
  decoding: greedy
results:
- id: r_mq_sq_none_tasr
  evaluation_id: ev_mq_sq_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '28.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sq_none_acc
  evaluation_id: ev_mq_sq_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '90.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_mq_none_tasr
  evaluation_id: ev_mq_mq_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '26.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_mq_none_acc
  evaluation_id: ev_mq_mq_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '89.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_aq_none_tasr
  evaluation_id: ev_mq_aq_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '21.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_aq_none_acc
  evaluation_id: ev_mq_aq_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '90.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_none_tasr
  evaluation_id: ev_mq_sl_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '52.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_none_acc
  evaluation_id: ev_mq_sl_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '83.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_none_tasr
  evaluation_id: ev_mq_ml_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '60.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_none_acc
  evaluation_id: ev_mq_ml_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '81.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sq_instr_tasr
  evaluation_id: ev_mq_sq_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '20.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sq_instr_acc
  evaluation_id: ev_mq_sq_instr_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '91.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_mq_instr_tasr
  evaluation_id: ev_mq_mq_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '18.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_mq_instr_acc
  evaluation_id: ev_mq_mq_instr_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '90.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_aq_instr_tasr
  evaluation_id: ev_mq_aq_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '11.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_aq_instr_acc
  evaluation_id: ev_mq_aq_instr_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '90.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_instr_tasr
  evaluation_id: ev_mq_sl_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '51.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_instr_acc
  evaluation_id: ev_mq_sl_instr_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '83.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_instr_tasr
  evaluation_id: ev_mq_ml_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '61.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_instr_acc
  evaluation_id: ev_mq_ml_instr_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '78.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sq_sand_tasr
  evaluation_id: ev_mq_sq_sand
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '23.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sq_sand_acc
  evaluation_id: ev_mq_sq_sand_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '90.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_mq_sand_tasr
  evaluation_id: ev_mq_mq_sand
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '25.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_mq_sand_acc
  evaluation_id: ev_mq_mq_sand_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '90.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_aq_sand_tasr
  evaluation_id: ev_mq_aq_sand
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '19.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_aq_sand_acc
  evaluation_id: ev_mq_aq_sand_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '90.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_sand_tasr
  evaluation_id: ev_mq_sl_sand
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '52.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_sand_acc
  evaluation_id: ev_mq_sl_sand_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '83.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_sand_tasr
  evaluation_id: ev_mq_ml_sand
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '55.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_sand_acc
  evaluation_id: ev_mq_ml_sand_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '78.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sq_delim_tasr
  evaluation_id: ev_mq_sq_delim
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '30.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sq_delim_acc
  evaluation_id: ev_mq_sq_delim_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '91.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_mq_delim_tasr
  evaluation_id: ev_mq_mq_delim
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '27.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_mq_delim_acc
  evaluation_id: ev_mq_mq_delim_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '90.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_aq_delim_tasr
  evaluation_id: ev_mq_aq_delim
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '18.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_aq_delim_acc
  evaluation_id: ev_mq_aq_delim_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '90.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_delim_tasr
  evaluation_id: ev_mq_sl_delim
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '53.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_delim_acc
  evaluation_id: ev_mq_sl_delim_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '83.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_delim_tasr
  evaluation_id: ev_mq_ml_delim
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '62.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_delim_acc
  evaluation_id: ev_mq_ml_delim_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '80.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_msa_tasr
  evaluation_id: ev_mq_sl_msa
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '48.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_sl_msa_acc
  evaluation_id: ev_mq_sl_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '76.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_msa_tasr
  evaluation_id: ev_mq_ml_msa
  quantity: R1
  coverage: direct
  metric_as_printed: TASR, all eight attacks
  value: '54.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_mq_ml_msa_acc
  evaluation_id: ev_mq_ml_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean accuracy
  value: '61.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_t2_mq_mmqa_none
  evaluation_id: ev_t2_mq_mmqa_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '25.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_mq_mmqa_instr
  evaluation_id: ev_t2_mq_mmqa_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '19.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_mq_mmmc_none
  evaluation_id: ev_t2_mq_mmmc_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '34.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_mq_mmmc_instr
  evaluation_id: ev_t2_mq_mmmc_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '26.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_mq_mmx_none
  evaluation_id: ev_t2_mq_mmx_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '36.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_mq_mmx_instr
  evaluation_id: ev_t2_mq_mmx_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '28.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_aq_mmqa_none
  evaluation_id: ev_t2_aq_mmqa_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '16.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_aq_mmqa_instr
  evaluation_id: ev_t2_aq_mmqa_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '13.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_aq_mmmc_none
  evaluation_id: ev_t2_aq_mmmc_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '23.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_aq_mmmc_instr
  evaluation_id: ev_t2_aq_mmmc_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '16.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_aq_mmx_none
  evaluation_id: ev_t2_aq_mmx_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '24.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_aq_mmx_instr
  evaluation_id: ev_t2_aq_mmx_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '21.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_ml_mmx_none
  evaluation_id: ev_t2_ml_mmx_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '43.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_ml_mmx_instr
  evaluation_id: ev_t2_ml_mmx_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '42.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_sq_mmx_none
  evaluation_id: ev_t2_sq_mmx_none
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '36.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_sq_mmx_instr
  evaluation_id: ev_t2_sq_mmx_instr
  quantity: R1
  coverage: direct
  metric_as_printed: TASR
  value: '33.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_mq_mpib_none
  evaluation_id: ev_t2_mq_mpib_none
  quantity: R1
  coverage: direct
  metric_as_printed: harmful answers (ASR2)
  value: '26.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_mq_mpib_instr
  evaluation_id: ev_t2_mq_mpib_instr
  quantity: R1
  coverage: direct
  metric_as_printed: harmful answers (ASR2)
  value: '26.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_ml_mpib_none
  evaluation_id: ev_t2_ml_mpib_none
  quantity: R1
  coverage: direct
  metric_as_printed: harmful answers (ASR2)
  value: '33.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_ml_mpib_instr
  evaluation_id: ev_t2_ml_mpib_instr
  quantity: R1
  coverage: direct
  metric_as_printed: harmful answers (ASR2)
  value: '34.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_sq_mpib_none
  evaluation_id: ev_t2_sq_mpib_none
  quantity: R1
  coverage: direct
  metric_as_printed: harmful answers (ASR2)
  value: '28.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_t2_sq_mpib_instr
  evaluation_id: ev_t2_sq_mpib_instr
  quantity: R1
  coverage: direct
  metric_as_printed: harmful answers (ASR2)
  value: '30.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
comparisons:
- id: c_mq_sq_instr
  left: ev_mq_sq_instr
  right: ev_mq_sq_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sq_instr_clean
  left: ev_mq_sq_instr_clean
  right: ev_mq_sq_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_mq_instr
  left: ev_mq_mq_instr
  right: ev_mq_mq_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_mq_instr_clean
  left: ev_mq_mq_instr_clean
  right: ev_mq_mq_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_aq_instr
  left: ev_mq_aq_instr
  right: ev_mq_aq_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_aq_instr_clean
  left: ev_mq_aq_instr_clean
  right: ev_mq_aq_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sl_instr
  left: ev_mq_sl_instr
  right: ev_mq_sl_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sl_instr_clean
  left: ev_mq_sl_instr_clean
  right: ev_mq_sl_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_ml_instr
  left: ev_mq_ml_instr
  right: ev_mq_ml_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_ml_instr_clean
  left: ev_mq_ml_instr_clean
  right: ev_mq_ml_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sq_sand
  left: ev_mq_sq_sand
  right: ev_mq_sq_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sq_sand_clean
  left: ev_mq_sq_sand_clean
  right: ev_mq_sq_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_mq_sand
  left: ev_mq_mq_sand
  right: ev_mq_mq_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_mq_sand_clean
  left: ev_mq_mq_sand_clean
  right: ev_mq_mq_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_aq_sand
  left: ev_mq_aq_sand
  right: ev_mq_aq_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_aq_sand_clean
  left: ev_mq_aq_sand_clean
  right: ev_mq_aq_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sl_sand
  left: ev_mq_sl_sand
  right: ev_mq_sl_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sl_sand_clean
  left: ev_mq_sl_sand_clean
  right: ev_mq_sl_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_ml_sand
  left: ev_mq_ml_sand
  right: ev_mq_ml_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_ml_sand_clean
  left: ev_mq_ml_sand_clean
  right: ev_mq_ml_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sq_delim
  left: ev_mq_sq_delim
  right: ev_mq_sq_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sq_delim_clean
  left: ev_mq_sq_delim_clean
  right: ev_mq_sq_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_mq_delim
  left: ev_mq_mq_delim
  right: ev_mq_mq_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_mq_delim_clean
  left: ev_mq_mq_delim_clean
  right: ev_mq_mq_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_aq_delim
  left: ev_mq_aq_delim
  right: ev_mq_aq_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_aq_delim_clean
  left: ev_mq_aq_delim_clean
  right: ev_mq_aq_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sl_delim
  left: ev_mq_sl_delim
  right: ev_mq_sl_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sl_delim_clean
  left: ev_mq_sl_delim_clean
  right: ev_mq_sl_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_ml_delim
  left: ev_mq_ml_delim
  right: ev_mq_ml_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_ml_delim_clean
  left: ev_mq_ml_delim_clean
  right: ev_mq_ml_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sl_msa
  left: ev_mq_sl_msa
  right: ev_mq_sl_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_sl_msa_clean
  left: ev_mq_sl_msa_clean
  right: ev_mq_sl_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_ml_msa
  left: ev_mq_ml_msa
  right: ev_mq_ml_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_mq_ml_msa_clean
  left: ev_mq_ml_msa_clean
  right: ev_mq_ml_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_t2_mq_mmqa
  left: ev_t2_mq_mmqa_instr
  right: ev_t2_mq_mmqa_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_mq_mmmc
  left: ev_t2_mq_mmmc_instr
  right: ev_t2_mq_mmmc_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_mq_mmx
  left: ev_t2_mq_mmx_instr
  right: ev_t2_mq_mmx_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_aq_mmqa
  left: ev_t2_aq_mmqa_instr
  right: ev_t2_aq_mmqa_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_aq_mmmc
  left: ev_t2_aq_mmmc_instr
  right: ev_t2_aq_mmmc_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_aq_mmx
  left: ev_t2_aq_mmx_instr
  right: ev_t2_aq_mmx_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_ml_mmx
  left: ev_t2_ml_mmx_instr
  right: ev_t2_ml_mmx_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_sq_mmx
  left: ev_t2_sq_mmx_instr
  right: ev_t2_sq_mmx_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_mq_mpib
  left: ev_t2_mq_mpib_instr
  right: ev_t2_mq_mpib_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_ml_mpib
  left: ev_t2_ml_mpib_instr
  right: ev_t2_ml_mpib_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t2_sq_mpib
  left: ev_t2_sq_mpib_instr
  right: ev_t2_sq_mpib_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
claims:
- id: k1
  text: existing defences help only partly and miss injections that look like ordinary clinical text
  location: abstract
  anchor: Existing defenses help only partly
  jobs:
  - R1
  supported_by:
  - c_mq_sq_instr
  - c_mq_mq_instr
  - c_mq_aq_instr
  - c_mq_sl_instr
  - c_mq_ml_instr
  - c_mq_sq_sand
  - c_mq_mq_sand
  - c_mq_aq_sand
  - c_mq_sl_sand
  - c_mq_ml_sand
  - c_mq_sq_delim
  - c_mq_mq_delim
  - c_mq_aq_delim
  - c_mq_sl_delim
  - c_mq_ml_delim
  - c_mq_sl_msa
  - c_mq_ml_msa
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1 and 2
    R2clean: Table 1 (clean accuracy)
    R2attack: not reported
schema_gaps:
- where: report
  note: most injections persuade rather than instruct (six of the eight MedQA attacks), so instruction-focused defences are tested partly outside their design; Meta-SecAlign lowers clean accuracy in MDAgents from 81.4% to
    61.9%

`pipette_arxiv2026` (2611.24362)

In [ ]:
%%writefile reports/records/pipette_arxiv2026.yaml
report:
  id: pipette_arxiv2026
  slug: '2611.24362'
  title: 'PIPETTE: Prompt Injection Purple-Teaming for Robust LLM Agents'
  first_author: Lipeng He
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2611.24362'
  version_read: arXiv v1
  first_date: '2026-11-18'
  read: F
  role: proposes
  extraction_scope: 'Table 6 (AgentDojo static, average over five attacks, and benign utility: None, Spotlighting, Sandwiching and PIPETTE on Qwen3-4B and Llama-3.1-8B), Tables 9 and 10 (averages: Meta-SecAlign, SecOPD and
    PIPETTE against None on AgentDojo and ASB) and Table 5 (TAP and GENETIC adaptive search); the Section 5.4 claim (PromptGuard, PIGuard, DataSentinel, MELON and CaMeL are detectors, masked re-execution and system-level isolation
    and are left out; the red-teaming tables test attackers)'
defences:
- id: pip_q4
  name: PIPETTE (adversarial training on PIPETTE-Red attacks with outcome RL and on-policy self-distillation toward a safe-prefix target)
  origin: this_report
  base_checkpoint: Qwen3-4B-Instruct-2507
  axis1:
  - chat_roles
  axis2:
  - adversarial_rl
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we introduce P IPETTE, a unified prompt
- id: pip_l8
  name: PIPETTE (adversarial training on PIPETTE-Red attacks with outcome RL and on-policy self-distillation toward a safe-prefix target)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - adversarial_rl
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: *id001
  anchor: we introduce P IPETTE, a unified prompt
- id: pip_q27
  name: PIPETTE (adversarial training on PIPETTE-Red attacks with outcome RL and on-policy self-distillation toward a safe-prefix target)
  origin: this_report
  base_checkpoint: Qwen3.6-27B
  axis1:
  - chat_roles
  axis2:
  - adversarial_rl
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: *id001
  anchor: we introduce P IPETTE, a unified prompt
- id: pip_oss
  name: PIPETTE (adversarial training on PIPETTE-Red attacks with outcome RL and on-policy self-distillation toward a safe-prefix target)
  origin: this_report
  base_checkpoint: GPT-OSS-20B (native instruction hierarchy)
  axis1:
  - chat_roles
  axis2:
  - adversarial_rl
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: *id001
  anchor: we introduce P IPETTE, a unified prompt
- id: spot_q4
  name: Spotlighting (observations wrapped in delimiters)
  origin: other_report
  base_checkpoint: Qwen3-4B-Instruct-2507
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 11
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: *id001
  anchor: wrap observations in delimiters or repeat
- id: sand_q4
  name: Sandwiching (user task repeated after each observation)
  origin: other_report
  base_checkpoint: Qwen3-4B-Instruct-2507
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 11
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: *id001
  anchor: wrap observations in delimiters or repeat
- id: spot_l8
  name: Spotlighting (observations wrapped in delimiters)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 11
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: *id001
  anchor: wrap observations in delimiters or repeat
- id: sand_l8
  name: Sandwiching (user task repeated after each observation)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 11
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: *id001
  anchor: wrap observations in delimiters or repeat
- id: msa_l8
  name: Meta-SecAlign (authors' released checkpoint)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: *id001
  anchor: Meta-SecAlign [15] use pref-
- id: secopd_q27
  name: SecOPD (authors' released checkpoint)
  origin: other_report
  base_checkpoint: Qwen3.6-27B
  axis1:
  - chat_roles
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: AgentDojo or ASB tool-calling agent; injections in tool outputs
  layout_compat: *id001
  anchor: SecOPD [21] uses on-policy distillation
evaluations:
- id: ev_adj_q4_none
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
  baseline_kind: off_the_shelf
- id: ev_adj_q4_none_bu
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: '97'
  baseline_kind: off_the_shelf
- id: ev_adj_q4_spot
  defence_id: spot_q4
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_q4_spot_bu
  defence_id: spot_q4
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_adj_q4_sand
  defence_id: sand_q4
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_q4_sand_bu
  defence_id: sand_q4
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_adj_q4_pip
  defence_id: pip_q4
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_q4_pip_bu
  defence_id: pip_q4
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_adj_l8_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
  baseline_kind: off_the_shelf
- id: ev_adj_l8_none_bu
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: '97'
  baseline_kind: off_the_shelf
- id: ev_adj_l8_spot
  defence_id: spot_l8
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_l8_spot_bu
  defence_id: spot_l8
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_adj_l8_sand
  defence_id: sand_l8
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_l8_sand_bu
  defence_id: sand_l8
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_adj_l8_pip
  defence_id: pip_l8
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_l8_pip_bu
  defence_id: pip_l8
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 12
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_adj_l8_msa
  defence_id: msa_l8
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_l8_msa_bu
  defence_id: msa_l8
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_adj_q27_none
  defence_id: none:Qwen3.6-27B
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
  baseline_kind: off_the_shelf
- id: ev_adj_q27_none_bu
  defence_id: none:Qwen3.6-27B
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: '97'
  baseline_kind: off_the_shelf
- id: ev_adj_q27_secopd
  defence_id: secopd_q27
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_q27_secopd_bu
  defence_id: secopd_q27
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_adj_q27_pip
  defence_id: pip_q27
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_q27_pip_bu
  defence_id: pip_q27
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_adj_oss_none
  defence_id: none:GPT-OSS-20B (native instruction hierarchy)
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
  baseline_kind: off_the_shelf
- id: ev_adj_oss_none_bu
  defence_id: none:GPT-OSS-20B (native instruction hierarchy)
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: '97'
  baseline_kind: off_the_shelf
- id: ev_adj_oss_pip
  defence_id: pip_oss
  benchmark: AgentDojo, five unseen static attacks (IgnorePrev, InjecAgent, ImportantMsgs, ToolKnow, ChatInject), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 420 task pairs per attack
- id: ev_adj_oss_pip_bu
  defence_id: pip_oss
  benchmark: AgentDojo, 97 clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: user-task check
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: '97'
- id: ev_asb_l8_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: ASB, five static attacks (Naive, Escape, Ignore, Completion, Combined), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 400 cases per attack
  baseline_kind: off_the_shelf
- id: ev_asb_l8_msa
  defence_id: msa_l8
  benchmark: ASB, five static attacks (Naive, Escape, Ignore, Completion, Combined), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 400 cases per attack
- id: ev_asb_l8_pip
  defence_id: pip_l8
  benchmark: ASB, five static attacks (Naive, Escape, Ignore, Completion, Combined), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 400 cases per attack
- id: ev_asb_q27_none
  defence_id: none:Qwen3.6-27B
  benchmark: ASB, five static attacks (Naive, Escape, Ignore, Completion, Combined), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 400 cases per attack
  baseline_kind: off_the_shelf
- id: ev_asb_q27_secopd
  defence_id: secopd_q27
  benchmark: ASB, five static attacks (Naive, Escape, Ignore, Completion, Combined), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 400 cases per attack
- id: ev_asb_q27_pip
  defence_id: pip_q27
  benchmark: ASB, five static attacks (Naive, Escape, Ignore, Completion, Combined), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 400 cases per attack
- id: ev_asb_oss_none
  defence_id: none:GPT-OSS-20B (native instruction hierarchy)
  benchmark: ASB, five static attacks (Naive, Escape, Ignore, Completion, Combined), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 400 cases per attack
  baseline_kind: off_the_shelf
- id: ev_asb_oss_pip
  defence_id: pip_oss
  benchmark: ASB, five static attacks (Naive, Escape, Ignore, Completion, Combined), average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 16
  decoding: greedy, temperature 0
  sample_size: 400 cases per attack
- id: ev_adtap_l8_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, 40 task pairs, TAP iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
  baseline_kind: off_the_shelf
- id: ev_adgen_l8_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, 40 task pairs, GENETIC iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
  baseline_kind: off_the_shelf
- id: ev_adtap_l8_msa
  defence_id: msa_l8
  benchmark: AgentDojo, 40 task pairs, TAP iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
- id: ev_adgen_l8_msa
  defence_id: msa_l8
  benchmark: AgentDojo, 40 task pairs, GENETIC iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
- id: ev_adtap_l8_pip
  defence_id: pip_l8
  benchmark: AgentDojo, 40 task pairs, TAP iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
- id: ev_adgen_l8_pip
  defence_id: pip_l8
  benchmark: AgentDojo, 40 task pairs, GENETIC iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
- id: ev_adtap_q27_none
  defence_id: none:Qwen3.6-27B
  benchmark: AgentDojo, 40 task pairs, TAP iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
  baseline_kind: off_the_shelf
- id: ev_adgen_q27_none
  defence_id: none:Qwen3.6-27B
  benchmark: AgentDojo, 40 task pairs, GENETIC iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
  baseline_kind: off_the_shelf
- id: ev_adtap_q27_secopd
  defence_id: secopd_q27
  benchmark: AgentDojo, 40 task pairs, TAP iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
- id: ev_adgen_q27_secopd
  defence_id: secopd_q27
  benchmark: AgentDojo, 40 task pairs, GENETIC iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
- id: ev_adtap_q27_pip
  defence_id: pip_q27
  benchmark: AgentDojo, 40 task pairs, TAP iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
- id: ev_adgen_q27_pip
  defence_id: pip_q27
  benchmark: AgentDojo, 40 task pairs, GENETIC iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
- id: ev_adtap_oss_none
  defence_id: none:GPT-OSS-20B (native instruction hierarchy)
  benchmark: AgentDojo, 40 task pairs, TAP iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
  baseline_kind: off_the_shelf
- id: ev_adgen_oss_none
  defence_id: none:GPT-OSS-20B (native instruction hierarchy)
  benchmark: AgentDojo, 40 task pairs, GENETIC iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
  baseline_kind: off_the_shelf
- id: ev_adtap_oss_pip
  defence_id: pip_oss
  benchmark: AgentDojo, 40 task pairs, TAP iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
- id: ev_adgen_oss_pip
  defence_id: pip_oss
  benchmark: AgentDojo, 40 task pairs, GENETIC iterative search against each evaluated policy
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: pipette_arxiv2026
  page: 11
  decoding: greedy, temperature 0
  sample_size: 40 task pairs
  attacker_budget: attacker and judge LLMs of 4B; stops at first success
results:
- id: r_adj_q4_none_asr
  evaluation_id: ev_adj_q4_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '33.29'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_none_ua
  evaluation_id: ev_adj_q4_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '44.48'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_none_bu
  evaluation_id: ev_adj_q4_none_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '61.86'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_spot_asr
  evaluation_id: ev_adj_q4_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '29.14'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_spot_ua
  evaluation_id: ev_adj_q4_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '43.33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_spot_bu
  evaluation_id: ev_adj_q4_spot_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '57.73'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_sand_asr
  evaluation_id: ev_adj_q4_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '17.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_sand_ua
  evaluation_id: ev_adj_q4_sand
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '49.14'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_sand_bu
  evaluation_id: ev_adj_q4_sand_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '60.82'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_pip_asr
  evaluation_id: ev_adj_q4_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_pip_ua
  evaluation_id: ev_adj_q4_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '57.86'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_q4_pip_bu
  evaluation_id: ev_adj_q4_pip_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '64.95'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_none_asr
  evaluation_id: ev_adj_l8_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '13.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_none_ua
  evaluation_id: ev_adj_l8_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '33.05'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_none_bu
  evaluation_id: ev_adj_l8_none_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '40.21'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_spot_asr
  evaluation_id: ev_adj_l8_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '11.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_spot_ua
  evaluation_id: ev_adj_l8_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '28.86'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_spot_bu
  evaluation_id: ev_adj_l8_spot_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '34.02'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_sand_asr
  evaluation_id: ev_adj_l8_sand
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '6.57'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_sand_ua
  evaluation_id: ev_adj_l8_sand
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '34.57'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_sand_bu
  evaluation_id: ev_adj_l8_sand_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '36.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_pip_asr
  evaluation_id: ev_adj_l8_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.43'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_pip_ua
  evaluation_id: ev_adj_l8_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '39.05'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_pip_bu
  evaluation_id: ev_adj_l8_pip_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '41.24'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 6
- id: r_adj_l8_msa_asr
  evaluation_id: ev_adj_l8_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.81'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_l8_msa_ua
  evaluation_id: ev_adj_l8_msa
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '32.57'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_l8_msa_bu
  evaluation_id: ev_adj_l8_msa_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '29.90'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_q27_none_asr
  evaluation_id: ev_adj_q27_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '17.71'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_q27_none_ua
  evaluation_id: ev_adj_q27_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '78.95'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_q27_none_bu
  evaluation_id: ev_adj_q27_none_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '93.81'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_q27_secopd_asr
  evaluation_id: ev_adj_q27_secopd
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.52'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_q27_secopd_ua
  evaluation_id: ev_adj_q27_secopd
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '77.05'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_q27_secopd_bu
  evaluation_id: ev_adj_q27_secopd_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '84.54'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_q27_pip_asr
  evaluation_id: ev_adj_q27_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_q27_pip_ua
  evaluation_id: ev_adj_q27_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '81.19'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_q27_pip_bu
  evaluation_id: ev_adj_q27_pip_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '92.78'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_oss_none_asr
  evaluation_id: ev_adj_oss_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '47.29'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_oss_none_ua
  evaluation_id: ev_adj_oss_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '25.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_oss_none_bu
  evaluation_id: ev_adj_oss_none_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '50.52'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_oss_pip_asr
  evaluation_id: ev_adj_oss_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.24'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_oss_pip_ua
  evaluation_id: ev_adj_oss_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '24.19'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_adj_oss_pip_bu
  evaluation_id: ev_adj_oss_pip_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility (BU)
  value: '49.48'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 9
- id: r_asb_l8_none_asr
  evaluation_id: ev_asb_l8_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '6.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_l8_none_ua
  evaluation_id: ev_asb_l8_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '48.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_l8_msa_asr
  evaluation_id: ev_asb_l8_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_l8_msa_ua
  evaluation_id: ev_asb_l8_msa
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '48.30'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_l8_pip_asr
  evaluation_id: ev_asb_l8_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_l8_pip_ua
  evaluation_id: ev_asb_l8_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '51.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_q27_none_asr
  evaluation_id: ev_asb_q27_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '21.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_q27_none_ua
  evaluation_id: ev_asb_q27_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '68.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_q27_secopd_asr
  evaluation_id: ev_asb_q27_secopd
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_q27_secopd_ua
  evaluation_id: ev_asb_q27_secopd
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '87.80'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_q27_pip_asr
  evaluation_id: ev_asb_q27_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_q27_pip_ua
  evaluation_id: ev_asb_q27_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '87.45'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_oss_none_asr
  evaluation_id: ev_asb_oss_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '22.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_oss_none_ua
  evaluation_id: ev_asb_oss_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '52.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_oss_pip_asr
  evaluation_id: ev_asb_oss_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_asb_oss_pip_ua
  evaluation_id: ev_asb_oss_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '51.80'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 10
- id: r_adtap_l8_none_asr
  evaluation_id: ev_adtap_l8_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '7.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_l8_none_ua
  evaluation_id: ev_adtap_l8_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '12.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_l8_none_asr
  evaluation_id: ev_adgen_l8_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '17.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_l8_none_ua
  evaluation_id: ev_adgen_l8_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '32.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_l8_msa_asr
  evaluation_id: ev_adtap_l8_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_l8_msa_ua
  evaluation_id: ev_adtap_l8_msa
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '15.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_l8_msa_asr
  evaluation_id: ev_adgen_l8_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '12.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_l8_msa_ua
  evaluation_id: ev_adgen_l8_msa
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '17.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_l8_pip_asr
  evaluation_id: ev_adtap_l8_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_l8_pip_ua
  evaluation_id: ev_adtap_l8_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '25.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_l8_pip_asr
  evaluation_id: ev_adgen_l8_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_l8_pip_ua
  evaluation_id: ev_adgen_l8_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '30.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_q27_none_asr
  evaluation_id: ev_adtap_q27_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '40.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_q27_none_ua
  evaluation_id: ev_adtap_q27_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '97.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_q27_none_asr
  evaluation_id: ev_adgen_q27_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '70.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_q27_none_ua
  evaluation_id: ev_adgen_q27_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '97.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_q27_secopd_asr
  evaluation_id: ev_adtap_q27_secopd
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '15.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_q27_secopd_ua
  evaluation_id: ev_adtap_q27_secopd
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '95.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_q27_secopd_asr
  evaluation_id: ev_adgen_q27_secopd
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '30.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_q27_secopd_ua
  evaluation_id: ev_adgen_q27_secopd
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '92.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_q27_pip_asr
  evaluation_id: ev_adtap_q27_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_q27_pip_ua
  evaluation_id: ev_adtap_q27_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '87.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_q27_pip_asr
  evaluation_id: ev_adgen_q27_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '5.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_q27_pip_ua
  evaluation_id: ev_adgen_q27_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '90.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_oss_none_asr
  evaluation_id: ev_adtap_oss_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '50.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_oss_none_ua
  evaluation_id: ev_adtap_oss_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '87.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_oss_none_asr
  evaluation_id: ev_adgen_oss_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '92.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_oss_none_ua
  evaluation_id: ev_adgen_oss_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '85.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_oss_pip_asr
  evaluation_id: ev_adtap_oss_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '5.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adtap_oss_pip_ua
  evaluation_id: ev_adtap_oss_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '85.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_oss_pip_asr
  evaluation_id: ev_adgen_oss_pip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '10.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
- id: r_adgen_oss_pip_ua
  evaluation_id: ev_adgen_oss_pip
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack (UA)
  value: '82.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
comparisons:
- id: c_adj_q4_spot
  left: ev_adj_q4_spot
  right: ev_adj_q4_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_q4_spot_bu
  left: ev_adj_q4_spot_bu
  right: ev_adj_q4_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_q4_sand
  left: ev_adj_q4_sand
  right: ev_adj_q4_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_q4_sand_bu
  left: ev_adj_q4_sand_bu
  right: ev_adj_q4_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_q4_pip
  left: ev_adj_q4_pip
  right: ev_adj_q4_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_q4_pip_bu
  left: ev_adj_q4_pip_bu
  right: ev_adj_q4_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_l8_spot
  left: ev_adj_l8_spot
  right: ev_adj_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_l8_spot_bu
  left: ev_adj_l8_spot_bu
  right: ev_adj_l8_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_l8_sand
  left: ev_adj_l8_sand
  right: ev_adj_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_l8_sand_bu
  left: ev_adj_l8_sand_bu
  right: ev_adj_l8_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_l8_pip
  left: ev_adj_l8_pip
  right: ev_adj_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_l8_pip_bu
  left: ev_adj_l8_pip_bu
  right: ev_adj_l8_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_l8_msa
  left: ev_adj_l8_msa
  right: ev_adj_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_l8_msa_bu
  left: ev_adj_l8_msa_bu
  right: ev_adj_l8_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_q27_secopd
  left: ev_adj_q27_secopd
  right: ev_adj_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_q27_secopd_bu
  left: ev_adj_q27_secopd_bu
  right: ev_adj_q27_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_q27_pip
  left: ev_adj_q27_pip
  right: ev_adj_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_q27_pip_bu
  left: ev_adj_q27_pip_bu
  right: ev_adj_q27_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_oss_pip
  left: ev_adj_oss_pip
  right: ev_adj_oss_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_oss_pip_bu
  left: ev_adj_oss_pip_bu
  right: ev_adj_oss_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adj_l8_pip_vs_msa
  left: ev_adj_l8_pip
  right: ev_adj_l8_msa
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: PIPETTE against Meta-SecAlign on the same base
  page: 16
- id: c_adj_q27_pip_vs_secopd
  left: ev_adj_q27_pip
  right: ev_adj_q27_secopd
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: PIPETTE against SecOPD on the same base
  page: 16
- id: c_asb_l8_msa
  left: ev_asb_l8_msa
  right: ev_asb_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_asb_l8_pip
  left: ev_asb_l8_pip
  right: ev_asb_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_asb_q27_secopd
  left: ev_asb_q27_secopd
  right: ev_asb_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_asb_q27_pip
  left: ev_asb_q27_pip
  right: ev_asb_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_asb_oss_pip
  left: ev_asb_oss_pip
  right: ev_asb_oss_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_adtap_l8_msa
  left: ev_adtap_l8_msa
  right: ev_adtap_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_adgen_l8_msa
  left: ev_adgen_l8_msa
  right: ev_adgen_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_adtap_l8_pip
  left: ev_adtap_l8_pip
  right: ev_adtap_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_adgen_l8_pip
  left: ev_adgen_l8_pip
  right: ev_adgen_l8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_adtap_q27_secopd
  left: ev_adtap_q27_secopd
  right: ev_adtap_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_adgen_q27_secopd
  left: ev_adgen_q27_secopd
  right: ev_adgen_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_adtap_q27_pip
  left: ev_adtap_q27_pip
  right: ev_adtap_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_adgen_q27_pip
  left: ev_adgen_q27_pip
  right: ev_adgen_q27_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_adtap_oss_pip
  left: ev_adtap_oss_pip
  right: ev_adtap_oss_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_adgen_oss_pip
  left: ev_adgen_oss_pip
  right: ev_adgen_oss_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
claims:
- id: k1
  text: PIPETTE has the lowest average attack success among the compared defences under static attacks on AgentDojo and ASB
  location: Section 5.4
  anchor: the lowest average ASR among the defenses compared under static attacks
  jobs:
  - R1
  supported_by:
  - c_adj_q4_pip
  - c_adj_l8_pip
  - c_adj_l8_pip_vs_msa
  - c_adj_q27_pip_vs_secopd
  - c_asb_l8_pip
  - c_asb_q27_pip
  - c_asb_oss_pip
  page: 11
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 5, 6, 9 and 10
    R2clean: Tables 6 and 9 (BU)
    R2attack: Tables 5, 6, 9 and 10 (UA)
schema_gaps:
- where: report
  note: PIPETTE is trained on AgentDyn and evaluated on AgentDojo and ASB; the static results are averages over five attack templates; Meta-SecAlign and SecOPD are evaluated only on the bases their authors released, so PIPETTE
    is compared with each on that base

`cbc_emnlp2026` (doi_10_18653_2026_emnlp_main_2303)

In [ ]:
%%writefile reports/records/cbc_emnlp2026.yaml
report:
  id: cbc_emnlp2026
  slug: doi_10_18653_2026_emnlp_main_2303
  title: 'Staying on Task: Defending Against Prompt Injection via Task Consistency Checking'
  first_author: Feiyue Xu
  venue: EMNLP 2026 (main conference)
  peer_reviewed: 'yes'
  doi: 10.18653/2026.emnlp-main.2303
  version_read: published version
  first_date: '2026-01-01'
  read: F
  role: proposes
  extraction_scope: Table 1 indirect-injection columns (SEP, OpenPrompt, InjecAgent, GCG, Fuzz) and Table 2 average utility, clean and under attack, for Base, Sandwich, StruQ, SecAlign, VerIH and CBC on three models; the abstract
    claim (PromptArmor is an LLM detector and is left out; the direct-injection columns are left out)
defences:
- id: sand_q25
  name: Sandwich (prompt engineering)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we use their released
- id: struq_q25
  name: StruQ (trained by the authors with the released data)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: secalign_q25
  name: SecAlign (trained by the authors with the released data)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: verih_q25
  name: VerIH (instruction-hierarchy RL, trained by the authors with the released data)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: cbc_q25
  name: CBC (SFT on teacher-written task-consistency reasoning, then GRPO with a failure-first reward)
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: GRPO optimization. We adopt online
- id: sand_l31
  name: Sandwich (prompt engineering)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: struq_l31
  name: StruQ (trained by the authors with the released data)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: secalign_l31
  name: SecAlign (trained by the authors with the released data)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: verih_l31
  name: VerIH (instruction-hierarchy RL, trained by the authors with the released data)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: cbc_l31
  name: CBC (SFT on teacher-written task-consistency reasoning, then GRPO with a failure-first reward)
  origin: this_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: GRPO optimization. We adopt online
- id: sand_q3
  name: Sandwich (prompt engineering)
  origin: other_report
  base_checkpoint: Qwen3-14B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: secalign_q3
  name: SecAlign (trained by the authors with the released data)
  origin: other_report
  base_checkpoint: Qwen3-14B
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: verih_q3
  name: VerIH (instruction-hierarchy RL, trained by the authors with the released data)
  origin: other_report
  base_checkpoint: Qwen3-14B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: we use their released
- id: cbc_q3
  name: CBC (SFT on teacher-written task-consistency reasoning, then GRPO with a failure-first reward)
  origin: this_report
  base_checkpoint: Qwen3-14B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  layout: system prompt, user request and external content; injected instruction inside the content
  layout_compat: *id001
  anchor: GRPO optimization. We adopt online
evaluations:
- id: ev_sep_q25_none
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_opi_q25_none
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ia_q25_none
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gcg_q25_none
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
  baseline_kind: off_the_shelf
- id: ev_fuzz_q25_none
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
  baseline_kind: off_the_shelf
- id: ev_uc_q25_none
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ua_q25_none
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_sep_q25_sand
  defence_id: sand_q25
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_q25_sand
  defence_id: sand_q25
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_q25_sand
  defence_id: sand_q25
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_q25_sand
  defence_id: sand_q25
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_q25_sand
  defence_id: sand_q25
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_q25_sand
  defence_id: sand_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_q25_sand
  defence_id: sand_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_q25_struq
  defence_id: struq_q25
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_q25_struq
  defence_id: struq_q25
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_q25_struq
  defence_id: struq_q25
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_q25_struq
  defence_id: struq_q25
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_q25_struq
  defence_id: struq_q25
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_q25_struq
  defence_id: struq_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_q25_struq
  defence_id: struq_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_q25_secalign
  defence_id: secalign_q25
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_q25_secalign
  defence_id: secalign_q25
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_q25_secalign
  defence_id: secalign_q25
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_q25_secalign
  defence_id: secalign_q25
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_q25_secalign
  defence_id: secalign_q25
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_q25_secalign
  defence_id: secalign_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_q25_secalign
  defence_id: secalign_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_q25_verih
  defence_id: verih_q25
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_q25_verih
  defence_id: verih_q25
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_q25_verih
  defence_id: verih_q25
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_q25_verih
  defence_id: verih_q25
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_q25_verih
  defence_id: verih_q25
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_q25_verih
  defence_id: verih_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_q25_verih
  defence_id: verih_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_q25_cbc
  defence_id: cbc_q25
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_q25_cbc
  defence_id: cbc_q25
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_q25_cbc
  defence_id: cbc_q25
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_q25_cbc
  defence_id: cbc_q25
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_q25_cbc
  defence_id: cbc_q25
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_q25_cbc
  defence_id: cbc_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_q25_cbc
  defence_id: cbc_q25
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_l31_none
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_opi_l31_none
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ia_l31_none
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gcg_l31_none
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
  baseline_kind: off_the_shelf
- id: ev_fuzz_l31_none
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
  baseline_kind: off_the_shelf
- id: ev_uc_l31_none
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ua_l31_none
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_sep_l31_sand
  defence_id: sand_l31
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_l31_sand
  defence_id: sand_l31
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_l31_sand
  defence_id: sand_l31
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_l31_sand
  defence_id: sand_l31
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_l31_sand
  defence_id: sand_l31
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_l31_sand
  defence_id: sand_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_l31_sand
  defence_id: sand_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_l31_struq
  defence_id: struq_l31
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_l31_struq
  defence_id: struq_l31
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_l31_struq
  defence_id: struq_l31
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_l31_struq
  defence_id: struq_l31
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_l31_struq
  defence_id: struq_l31
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_l31_struq
  defence_id: struq_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_l31_struq
  defence_id: struq_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_l31_secalign
  defence_id: secalign_l31
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_l31_secalign
  defence_id: secalign_l31
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_l31_secalign
  defence_id: secalign_l31
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_l31_secalign
  defence_id: secalign_l31
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_l31_secalign
  defence_id: secalign_l31
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_l31_secalign
  defence_id: secalign_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_l31_secalign
  defence_id: secalign_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_l31_verih
  defence_id: verih_l31
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_l31_verih
  defence_id: verih_l31
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_l31_verih
  defence_id: verih_l31
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_l31_verih
  defence_id: verih_l31
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_l31_verih
  defence_id: verih_l31
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_l31_verih
  defence_id: verih_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_l31_verih
  defence_id: verih_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_l31_cbc
  defence_id: cbc_l31
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_l31_cbc
  defence_id: cbc_l31
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_l31_cbc
  defence_id: cbc_l31
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_l31_cbc
  defence_id: cbc_l31
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_l31_cbc
  defence_id: cbc_l31
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_l31_cbc
  defence_id: cbc_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_l31_cbc
  defence_id: cbc_l31
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_q3_none
  defence_id: none:Qwen3-14B
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_opi_q3_none
  defence_id: none:Qwen3-14B
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ia_q3_none
  defence_id: none:Qwen3-14B
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gcg_q3_none
  defence_id: none:Qwen3-14B
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
  baseline_kind: off_the_shelf
- id: ev_fuzz_q3_none
  defence_id: none:Qwen3-14B
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
  baseline_kind: off_the_shelf
- id: ev_uc_q3_none
  defence_id: none:Qwen3-14B
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ua_q3_none
  defence_id: none:Qwen3-14B
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_sep_q3_sand
  defence_id: sand_q3
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_q3_sand
  defence_id: sand_q3
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_q3_sand
  defence_id: sand_q3
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_q3_sand
  defence_id: sand_q3
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_q3_sand
  defence_id: sand_q3
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_q3_sand
  defence_id: sand_q3
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_q3_sand
  defence_id: sand_q3
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_q3_secalign
  defence_id: secalign_q3
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_q3_secalign
  defence_id: secalign_q3
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_q3_secalign
  defence_id: secalign_q3
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_q3_secalign
  defence_id: secalign_q3
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_q3_secalign
  defence_id: secalign_q3
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_q3_secalign
  defence_id: secalign_q3
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_q3_secalign
  defence_id: secalign_q3
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_q3_verih
  defence_id: verih_q3
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_q3_verih
  defence_id: verih_q3
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_q3_verih
  defence_id: verih_q3
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_q3_verih
  defence_id: verih_q3
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_q3_verih
  defence_id: verih_q3
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_q3_verih
  defence_id: verih_q3
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_q3_verih
  defence_id: verih_q3
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sep_q3_cbc
  defence_id: cbc_q3
  benchmark: SEP (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_opi_q3_cbc
  defence_id: cbc_q3
  benchmark: OpenPrompt (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_ia_q3_cbc
  defence_id: cbc_q3
  benchmark: InjecAgent (indirect prompt injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
- id: ev_gcg_q3_cbc
  defence_id: cbc_q3
  benchmark: GCG on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: white_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: GCG optimised on each evaluated model
- id: ev_fuzz_q3_cbc
  defence_id: cbc_q3
  benchmark: fuzzing on indirect injection (indirect prompt injection)
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark rule; defence success when the injected goal is not achieved
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 6
  decoding: not reported
  attacker_budget: fuzzing attack run against each evaluated model
- id: ev_uc_q3_cbc
  defence_id: cbc_q3
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval, clean inputs, average
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
- id: ev_ua_q3_cbc
  defence_id: cbc_q3
  benchmark: MMLU, ARC, BBQ, GSM8K and AlpacaEval with a combined injection attack, average
  attacker_type: static
  attacker_access: black_box
  judge: benchmark scoring
  layout_same_as_defence: 'yes'
  measured_by: cbc_emnlp2026
  page: 7
  decoding: not reported
results:
- id: r_sep_q25_none
  evaluation_id: ev_sep_q25_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '27.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q25_none
  evaluation_id: ev_opi_q25_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '42.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q25_none
  evaluation_id: ev_ia_q25_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '64.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q25_none
  evaluation_id: ev_gcg_q25_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '0.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q25_none
  evaluation_id: ev_fuzz_q25_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '0.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q25_none
  evaluation_id: ev_uc_q25_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '71.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q25_none
  evaluation_id: ev_ua_q25_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '1.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q25_sand
  evaluation_id: ev_sep_q25_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '36.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q25_sand
  evaluation_id: ev_opi_q25_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '68.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q25_sand
  evaluation_id: ev_ia_q25_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '64.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q25_sand
  evaluation_id: ev_gcg_q25_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '2.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q25_sand
  evaluation_id: ev_fuzz_q25_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '0.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q25_sand
  evaluation_id: ev_uc_q25_sand
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '70.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q25_sand
  evaluation_id: ev_ua_q25_sand
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '50.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q25_struq
  evaluation_id: ev_sep_q25_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '95.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q25_struq
  evaluation_id: ev_opi_q25_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q25_struq
  evaluation_id: ev_ia_q25_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '99.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q25_struq
  evaluation_id: ev_gcg_q25_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '63.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q25_struq
  evaluation_id: ev_fuzz_q25_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '93.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q25_struq
  evaluation_id: ev_uc_q25_struq
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '64.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q25_struq
  evaluation_id: ev_ua_q25_struq
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '61.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q25_secalign
  evaluation_id: ev_sep_q25_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '95.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q25_secalign
  evaluation_id: ev_opi_q25_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '98.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q25_secalign
  evaluation_id: ev_ia_q25_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '86.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q25_secalign
  evaluation_id: ev_gcg_q25_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '69.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q25_secalign
  evaluation_id: ev_fuzz_q25_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '93.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q25_secalign
  evaluation_id: ev_uc_q25_secalign
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '71.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q25_secalign
  evaluation_id: ev_ua_q25_secalign
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '65.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q25_verih
  evaluation_id: ev_sep_q25_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '56.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q25_verih
  evaluation_id: ev_opi_q25_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '76.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q25_verih
  evaluation_id: ev_ia_q25_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '56.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q25_verih
  evaluation_id: ev_gcg_q25_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '4.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q25_verih
  evaluation_id: ev_fuzz_q25_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '10.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q25_verih
  evaluation_id: ev_uc_q25_verih
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '68.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q25_verih
  evaluation_id: ev_ua_q25_verih
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '34.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q25_cbc
  evaluation_id: ev_sep_q25_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '96.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q25_cbc
  evaluation_id: ev_opi_q25_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '98.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q25_cbc
  evaluation_id: ev_ia_q25_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q25_cbc
  evaluation_id: ev_gcg_q25_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '94.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q25_cbc
  evaluation_id: ev_fuzz_q25_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '97.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q25_cbc
  evaluation_id: ev_uc_q25_cbc
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '74.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q25_cbc
  evaluation_id: ev_ua_q25_cbc
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '71.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_l31_none
  evaluation_id: ev_sep_l31_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '35.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_l31_none
  evaluation_id: ev_opi_l31_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '56.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_l31_none
  evaluation_id: ev_ia_l31_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '54.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_l31_none
  evaluation_id: ev_gcg_l31_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '9.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_l31_none
  evaluation_id: ev_fuzz_l31_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '0.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_l31_none
  evaluation_id: ev_uc_l31_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '59.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_l31_none
  evaluation_id: ev_ua_l31_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '0.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_l31_sand
  evaluation_id: ev_sep_l31_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '49.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_l31_sand
  evaluation_id: ev_opi_l31_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '79.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_l31_sand
  evaluation_id: ev_ia_l31_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '64.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_l31_sand
  evaluation_id: ev_gcg_l31_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '15.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_l31_sand
  evaluation_id: ev_fuzz_l31_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '0.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_l31_sand
  evaluation_id: ev_uc_l31_sand
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '59.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_l31_sand
  evaluation_id: ev_ua_l31_sand
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '13.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_l31_struq
  evaluation_id: ev_sep_l31_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '93.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_l31_struq
  evaluation_id: ev_opi_l31_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '99.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_l31_struq
  evaluation_id: ev_ia_l31_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '99.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_l31_struq
  evaluation_id: ev_gcg_l31_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '95.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_l31_struq
  evaluation_id: ev_fuzz_l31_struq
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '91.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_l31_struq
  evaluation_id: ev_uc_l31_struq
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '55.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_l31_struq
  evaluation_id: ev_ua_l31_struq
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '53.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_l31_secalign
  evaluation_id: ev_sep_l31_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '95.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_l31_secalign
  evaluation_id: ev_opi_l31_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '99.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_l31_secalign
  evaluation_id: ev_ia_l31_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '99.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_l31_secalign
  evaluation_id: ev_gcg_l31_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '82.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_l31_secalign
  evaluation_id: ev_fuzz_l31_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_l31_secalign
  evaluation_id: ev_uc_l31_secalign
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '59.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_l31_secalign
  evaluation_id: ev_ua_l31_secalign
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '53.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_l31_verih
  evaluation_id: ev_sep_l31_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '51.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_l31_verih
  evaluation_id: ev_opi_l31_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '91.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_l31_verih
  evaluation_id: ev_ia_l31_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '91.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_l31_verih
  evaluation_id: ev_gcg_l31_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '67.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_l31_verih
  evaluation_id: ev_fuzz_l31_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '74.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_l31_verih
  evaluation_id: ev_uc_l31_verih
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '56.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_l31_verih
  evaluation_id: ev_ua_l31_verih
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '25.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_l31_cbc
  evaluation_id: ev_sep_l31_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '97.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_l31_cbc
  evaluation_id: ev_opi_l31_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '94.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_l31_cbc
  evaluation_id: ev_ia_l31_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_l31_cbc
  evaluation_id: ev_gcg_l31_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '91.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_l31_cbc
  evaluation_id: ev_fuzz_l31_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '94.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_l31_cbc
  evaluation_id: ev_uc_l31_cbc
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '72.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_l31_cbc
  evaluation_id: ev_ua_l31_cbc
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '63.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q3_none
  evaluation_id: ev_sep_q3_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '26.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q3_none
  evaluation_id: ev_opi_q3_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '53.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q3_none
  evaluation_id: ev_ia_q3_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '92.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q3_none
  evaluation_id: ev_gcg_q3_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '9.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q3_none
  evaluation_id: ev_fuzz_q3_none
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '4.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q3_none
  evaluation_id: ev_uc_q3_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '87.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q3_none
  evaluation_id: ev_ua_q3_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '54.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q3_sand
  evaluation_id: ev_sep_q3_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '35.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q3_sand
  evaluation_id: ev_opi_q3_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '84.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q3_sand
  evaluation_id: ev_ia_q3_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '88.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q3_sand
  evaluation_id: ev_gcg_q3_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '15.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q3_sand
  evaluation_id: ev_fuzz_q3_sand
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '18.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q3_sand
  evaluation_id: ev_uc_q3_sand
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '87.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q3_sand
  evaluation_id: ev_ua_q3_sand
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '67.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q3_secalign
  evaluation_id: ev_sep_q3_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '42.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q3_secalign
  evaluation_id: ev_opi_q3_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '93.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q3_secalign
  evaluation_id: ev_ia_q3_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '93.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q3_secalign
  evaluation_id: ev_gcg_q3_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '89.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q3_secalign
  evaluation_id: ev_fuzz_q3_secalign
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '93.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q3_secalign
  evaluation_id: ev_uc_q3_secalign
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '87.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q3_secalign
  evaluation_id: ev_ua_q3_secalign
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '71.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q3_verih
  evaluation_id: ev_sep_q3_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '28.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q3_verih
  evaluation_id: ev_opi_q3_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '61.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q3_verih
  evaluation_id: ev_ia_q3_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '92.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q3_verih
  evaluation_id: ev_gcg_q3_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '14.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q3_verih
  evaluation_id: ev_fuzz_q3_verih
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '26.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q3_verih
  evaluation_id: ev_uc_q3_verih
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '87.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q3_verih
  evaluation_id: ev_ua_q3_verih
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '58.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sep_q3_cbc
  evaluation_id: ev_sep_q3_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '99.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_opi_q3_cbc
  evaluation_id: ev_opi_q3_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '98.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_ia_q3_cbc
  evaluation_id: ev_ia_q3_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '99.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_gcg_q3_cbc
  evaluation_id: ev_gcg_q3_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '91.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_fuzz_q3_cbc
  evaluation_id: ev_fuzz_q3_cbc
  quantity: R1
  coverage: direct
  metric_as_printed: defence success rate (DSR)
  value: '97.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_uc_q3_cbc
  evaluation_id: ev_uc_q3_cbc
  quantity: R2clean
  coverage: direct
  metric_as_printed: average utility, clean
  value: '84.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_ua_q3_cbc
  evaluation_id: ev_ua_q3_cbc
  quantity: R2attack
  coverage: direct
  metric_as_printed: average utility under the combined attack
  value: '81.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
comparisons:
- id: c_sep_q25_sand
  left: ev_sep_q25_sand
  right: ev_sep_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_q25_sand
  left: ev_opi_q25_sand
  right: ev_opi_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_q25_sand
  left: ev_ia_q25_sand
  right: ev_ia_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_q25_sand
  left: ev_gcg_q25_sand
  right: ev_gcg_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_q25_sand
  left: ev_fuzz_q25_sand
  right: ev_fuzz_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_q25_sand
  left: ev_uc_q25_sand
  right: ev_uc_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_q25_sand
  left: ev_ua_q25_sand
  right: ev_ua_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_q25_struq
  left: ev_sep_q25_struq
  right: ev_sep_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_q25_struq
  left: ev_opi_q25_struq
  right: ev_opi_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_q25_struq
  left: ev_ia_q25_struq
  right: ev_ia_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_q25_struq
  left: ev_gcg_q25_struq
  right: ev_gcg_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_q25_struq
  left: ev_fuzz_q25_struq
  right: ev_fuzz_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_q25_struq
  left: ev_uc_q25_struq
  right: ev_uc_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_q25_struq
  left: ev_ua_q25_struq
  right: ev_ua_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_q25_secalign
  left: ev_sep_q25_secalign
  right: ev_sep_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_q25_secalign
  left: ev_opi_q25_secalign
  right: ev_opi_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_q25_secalign
  left: ev_ia_q25_secalign
  right: ev_ia_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_q25_secalign
  left: ev_gcg_q25_secalign
  right: ev_gcg_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_q25_secalign
  left: ev_fuzz_q25_secalign
  right: ev_fuzz_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_q25_secalign
  left: ev_uc_q25_secalign
  right: ev_uc_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_q25_secalign
  left: ev_ua_q25_secalign
  right: ev_ua_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_q25_verih
  left: ev_sep_q25_verih
  right: ev_sep_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_q25_verih
  left: ev_opi_q25_verih
  right: ev_opi_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_q25_verih
  left: ev_ia_q25_verih
  right: ev_ia_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_q25_verih
  left: ev_gcg_q25_verih
  right: ev_gcg_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_q25_verih
  left: ev_fuzz_q25_verih
  right: ev_fuzz_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_q25_verih
  left: ev_uc_q25_verih
  right: ev_uc_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_q25_verih
  left: ev_ua_q25_verih
  right: ev_ua_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_q25_cbc
  left: ev_sep_q25_cbc
  right: ev_sep_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_q25_cbc
  left: ev_opi_q25_cbc
  right: ev_opi_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_q25_cbc
  left: ev_ia_q25_cbc
  right: ev_ia_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_q25_cbc
  left: ev_gcg_q25_cbc
  right: ev_gcg_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_q25_cbc
  left: ev_fuzz_q25_cbc
  right: ev_fuzz_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_q25_cbc
  left: ev_uc_q25_cbc
  right: ev_uc_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_q25_cbc
  left: ev_ua_q25_cbc
  right: ev_ua_q25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_l31_sand
  left: ev_sep_l31_sand
  right: ev_sep_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_l31_sand
  left: ev_opi_l31_sand
  right: ev_opi_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_l31_sand
  left: ev_ia_l31_sand
  right: ev_ia_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_l31_sand
  left: ev_gcg_l31_sand
  right: ev_gcg_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_l31_sand
  left: ev_fuzz_l31_sand
  right: ev_fuzz_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_l31_sand
  left: ev_uc_l31_sand
  right: ev_uc_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_l31_sand
  left: ev_ua_l31_sand
  right: ev_ua_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_l31_struq
  left: ev_sep_l31_struq
  right: ev_sep_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_l31_struq
  left: ev_opi_l31_struq
  right: ev_opi_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_l31_struq
  left: ev_ia_l31_struq
  right: ev_ia_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_l31_struq
  left: ev_gcg_l31_struq
  right: ev_gcg_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_l31_struq
  left: ev_fuzz_l31_struq
  right: ev_fuzz_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_l31_struq
  left: ev_uc_l31_struq
  right: ev_uc_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_l31_struq
  left: ev_ua_l31_struq
  right: ev_ua_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_l31_secalign
  left: ev_sep_l31_secalign
  right: ev_sep_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_l31_secalign
  left: ev_opi_l31_secalign
  right: ev_opi_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_l31_secalign
  left: ev_ia_l31_secalign
  right: ev_ia_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_l31_secalign
  left: ev_gcg_l31_secalign
  right: ev_gcg_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_l31_secalign
  left: ev_fuzz_l31_secalign
  right: ev_fuzz_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_l31_secalign
  left: ev_uc_l31_secalign
  right: ev_uc_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_l31_secalign
  left: ev_ua_l31_secalign
  right: ev_ua_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_l31_verih
  left: ev_sep_l31_verih
  right: ev_sep_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_l31_verih
  left: ev_opi_l31_verih
  right: ev_opi_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_l31_verih
  left: ev_ia_l31_verih
  right: ev_ia_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_l31_verih
  left: ev_gcg_l31_verih
  right: ev_gcg_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_l31_verih
  left: ev_fuzz_l31_verih
  right: ev_fuzz_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_l31_verih
  left: ev_uc_l31_verih
  right: ev_uc_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_l31_verih
  left: ev_ua_l31_verih
  right: ev_ua_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_l31_cbc
  left: ev_sep_l31_cbc
  right: ev_sep_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_l31_cbc
  left: ev_opi_l31_cbc
  right: ev_opi_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_l31_cbc
  left: ev_ia_l31_cbc
  right: ev_ia_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_l31_cbc
  left: ev_gcg_l31_cbc
  right: ev_gcg_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_l31_cbc
  left: ev_fuzz_l31_cbc
  right: ev_fuzz_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_l31_cbc
  left: ev_uc_l31_cbc
  right: ev_uc_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_l31_cbc
  left: ev_ua_l31_cbc
  right: ev_ua_l31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_q3_sand
  left: ev_sep_q3_sand
  right: ev_sep_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_q3_sand
  left: ev_opi_q3_sand
  right: ev_opi_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_q3_sand
  left: ev_ia_q3_sand
  right: ev_ia_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_q3_sand
  left: ev_gcg_q3_sand
  right: ev_gcg_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_q3_sand
  left: ev_fuzz_q3_sand
  right: ev_fuzz_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_q3_sand
  left: ev_uc_q3_sand
  right: ev_uc_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_q3_sand
  left: ev_ua_q3_sand
  right: ev_ua_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_q3_secalign
  left: ev_sep_q3_secalign
  right: ev_sep_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_q3_secalign
  left: ev_opi_q3_secalign
  right: ev_opi_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_q3_secalign
  left: ev_ia_q3_secalign
  right: ev_ia_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_q3_secalign
  left: ev_gcg_q3_secalign
  right: ev_gcg_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_q3_secalign
  left: ev_fuzz_q3_secalign
  right: ev_fuzz_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_q3_secalign
  left: ev_uc_q3_secalign
  right: ev_uc_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_q3_secalign
  left: ev_ua_q3_secalign
  right: ev_ua_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_q3_verih
  left: ev_sep_q3_verih
  right: ev_sep_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_q3_verih
  left: ev_opi_q3_verih
  right: ev_opi_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_q3_verih
  left: ev_ia_q3_verih
  right: ev_ia_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_q3_verih
  left: ev_gcg_q3_verih
  right: ev_gcg_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_q3_verih
  left: ev_fuzz_q3_verih
  right: ev_fuzz_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_q3_verih
  left: ev_uc_q3_verih
  right: ev_uc_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_q3_verih
  left: ev_ua_q3_verih
  right: ev_ua_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_q3_cbc
  left: ev_sep_q3_cbc
  right: ev_sep_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_opi_q3_cbc
  left: ev_opi_q3_cbc
  right: ev_opi_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_ia_q3_cbc
  left: ev_ia_q3_cbc
  right: ev_ia_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg_q3_cbc
  left: ev_gcg_q3_cbc
  right: ev_gcg_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_fuzz_q3_cbc
  left: ev_fuzz_q3_cbc
  right: ev_fuzz_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_uc_q3_cbc
  left: ev_uc_q3_cbc
  right: ev_uc_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ua_q3_cbc
  left: ev_ua_q3_cbc
  right: ev_ua_q3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
claims:
- id: k1
  text: CBC gives strong and consistent robustness across indirect and direct attacks while preserving utility
  location: Section 4.2
  anchor: CBC achieves strong and consistent robustness across both settings
  jobs:
  - R1
  - R2
  supported_by:
  - c_sep_q25_cbc
  - c_opi_q25_cbc
  - c_ia_q25_cbc
  - c_gcg_q25_cbc
  - c_fuzz_q25_cbc
  - c_sep_l31_cbc
  - c_opi_l31_cbc
  - c_ia_l31_cbc
  - c_gcg_l31_cbc
  - c_fuzz_l31_cbc
  - c_sep_q3_cbc
  - c_opi_q3_cbc
  - c_ia_q3_cbc
  - c_gcg_q3_cbc
  - c_fuzz_q3_cbc
  - c_ua_q25_cbc
  - c_ua_l31_cbc
  - c_ua_q3_cbc
  page: 6
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (defence success rate, indirect columns)
    R2clean: Table 2 (clean average)
    R2attack: Table 2 (attack average)
schema_gaps:
- where: report
  note: results are defence success rates (higher is better) averaged over three runs; StruQ is omitted on Qwen3-14B because its released data do not fit the thinking format; the first date is the proceedings year as recorded
    at screening

`rui_preprints2026` (doi_10_20944_preprints202603_1023_v1)

In [ ]:
%%writefile reports/records/rui_preprints2026.yaml
report:
  id: rui_preprints2026
  slug: doi_10_20944_preprints202603_1023_v1
  title: 'Real User Instruction: Black-Box Instruction Authentication Middleware Against Indirect Prompt Injection'
  first_author: Jingtang Luo
  venue: Preprints.org preprint
  peer_reviewed: 'no'
  doi: 10.20944/preprints202603.1023.v1
  version_read: v1 (13 March 2026)
  first_date: '2026-03-13'
  read: F
  role: proposes
  code_release: https://github.com/lambblue/rui
  extraction_scope: 'Tables 2 to 5: no defence, static delimiter, randomized delimiter and RUI on five task scenarios (ASR and benign performance preservation), the split/encoded, spoofing and five-turn spoofing attacks, the
    cross-model table and the two mechanism ablations; the abstract claim (the user-mode ablation rows are not coded)'
defences:
- id: stat_gfm
  name: Static data delimiter (negative spotlighting)
  origin: other_report
  base_checkpoint: Gemini 3 Flash (medium thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Static Data Delimiters and Randomized
- id: rand_gfm
  name: Randomized (dynamic) data delimiter
  origin: other_report
  base_checkpoint: Gemini 3 Flash (medium thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: Static Data Delimiters and Randomized
- id: rui_gfm
  name: RUI (user instruction wrapped in a session-key JSON schema, mandatory detect-and-deny preamble, key rotation every turn)
  origin: this_report
  base_checkpoint: Gemini 3 Flash (medium thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: 'The Privileged Channel: A cryptographic-style encapsulation schema'
- id: stat_gfl
  name: Static data delimiter (negative spotlighting)
  origin: other_report
  base_checkpoint: Gemini 3 Flash (low thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: Static Data Delimiters and Randomized
- id: rand_gfl
  name: Randomized (dynamic) data delimiter
  origin: other_report
  base_checkpoint: Gemini 3 Flash (low thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: Static Data Delimiters and Randomized
- id: rui_gfl
  name: RUI (user instruction wrapped in a session-key JSON schema, mandatory detect-and-deny preamble, key rotation every turn)
  origin: this_report
  base_checkpoint: Gemini 3 Flash (low thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: 'The Privileged Channel: A cryptographic-style encapsulation schema'
- id: stat_gp
  name: Static data delimiter (negative spotlighting)
  origin: other_report
  base_checkpoint: Gemini 3.1 Pro (high thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: Static Data Delimiters and Randomized
- id: rand_gp
  name: Randomized (dynamic) data delimiter
  origin: other_report
  base_checkpoint: Gemini 3.1 Pro (high thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: Static Data Delimiters and Randomized
- id: rui_gp
  name: RUI (user instruction wrapped in a session-key JSON schema, mandatory detect-and-deny preamble, key rotation every turn)
  origin: this_report
  base_checkpoint: Gemini 3.1 Pro (high thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: 'The Privileged Channel: A cryptographic-style encapsulation schema'
- id: stat_glm
  name: Static data delimiter (negative spotlighting)
  origin: other_report
  base_checkpoint: GLM-5 (high thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: Static Data Delimiters and Randomized
- id: rand_glm
  name: Randomized (dynamic) data delimiter
  origin: other_report
  base_checkpoint: GLM-5 (high thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: Static Data Delimiters and Randomized
- id: rui_glm
  name: RUI (user instruction wrapped in a session-key JSON schema, mandatory detect-and-deny preamble, key rotation every turn)
  origin: this_report
  base_checkpoint: GLM-5 (high thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: 'The Privileged Channel: A cryptographic-style encapsulation schema'
- id: stat_g54
  name: Static data delimiter (negative spotlighting)
  origin: other_report
  base_checkpoint: GPT-5.4 (high thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: Static Data Delimiters and Randomized
- id: rand_g54
  name: Randomized (dynamic) data delimiter
  origin: other_report
  base_checkpoint: GPT-5.4 (high thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: Static Data Delimiters and Randomized
- id: rui_g54
  name: RUI (user instruction wrapped in a session-key JSON schema, mandatory detect-and-deny preamble, key rotation every turn)
  origin: this_report
  base_checkpoint: GPT-5.4 (high thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: 'The Privileged Channel: A cryptographic-style encapsulation schema'
- id: nopc_gfm
  name: RUI without the privileged channel (data wrapped instead of the instruction)
  origin: this_report
  base_checkpoint: Gemini 3 Flash (medium thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 21
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: We consider 3 mechanisms to evaluate
- id: noeai_gfm
  name: RUI without explicit adversarial identification (no listing of unauthorized instructions)
  origin: this_report
  base_checkpoint: Gemini 3 Flash (medium thinking)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 21
  layout: system prompt and user prompt rewritten by the middleware; external data in the user prompt
  layout_compat: *id001
  anchor: We consider 3 mechanisms to evaluate
evaluations:
- id: ev_dir_gfm_none_abstract
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_dir_gfm_none_code
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_dir_gfm_none_email
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_dir_gfm_none_report
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_dir_gfm_none_table
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_dir_gfm_stat_abstract
  defence_id: stat_gfm
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_stat_code
  defence_id: stat_gfm
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_stat_email
  defence_id: stat_gfm
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_stat_report
  defence_id: stat_gfm
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_stat_table
  defence_id: stat_gfm
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rand_abstract
  defence_id: rand_gfm
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rand_code
  defence_id: rand_gfm
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rand_email
  defence_id: rand_gfm
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rand_report
  defence_id: rand_gfm
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rand_table
  defence_id: rand_gfm
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rui_abstract
  defence_id: rui_gfm
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rui_code
  defence_id: rui_gfm
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rui_email
  defence_id: rui_gfm
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rui_report
  defence_id: rui_gfm
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_dir_gfm_rui_table
  defence_id: rui_gfm
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_stat_abstract
  defence_id: stat_gfm
  benchmark: RUI test set, abstract scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_stat_code
  defence_id: stat_gfm
  benchmark: RUI test set, code scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_stat_email
  defence_id: stat_gfm
  benchmark: RUI test set, email scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_stat_report
  defence_id: stat_gfm
  benchmark: RUI test set, report scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_stat_table
  defence_id: stat_gfm
  benchmark: RUI test set, table scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rand_abstract
  defence_id: rand_gfm
  benchmark: RUI test set, abstract scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rand_code
  defence_id: rand_gfm
  benchmark: RUI test set, code scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rand_email
  defence_id: rand_gfm
  benchmark: RUI test set, email scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rand_report
  defence_id: rand_gfm
  benchmark: RUI test set, report scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rand_table
  defence_id: rand_gfm
  benchmark: RUI test set, table scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rui_abstract
  defence_id: rui_gfm
  benchmark: RUI test set, abstract scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rui_code
  defence_id: rui_gfm
  benchmark: RUI test set, code scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rui_email
  defence_id: rui_gfm
  benchmark: RUI test set, email scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rui_report
  defence_id: rui_gfm
  benchmark: RUI test set, report scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_bpp_rui_table
  defence_id: rui_gfm
  benchmark: RUI test set, table scenario
  attacker_type: static
  attacker_access: black_box
  judge: 'Gemini-3.1-Pro judge: user command fulfilled as well as by the undefended model'
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 16
  decoding: not reported
- id: ev_split_gfm_none_abstract
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, abstract scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_split_gfm_none_code
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, code scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_split_gfm_none_email
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, email scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_split_gfm_none_report
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, report scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_split_gfm_none_table
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, table scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_split_gfm_stat_abstract
  defence_id: stat_gfm
  benchmark: RUI test set, abstract scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_stat_code
  defence_id: stat_gfm
  benchmark: RUI test set, code scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_stat_email
  defence_id: stat_gfm
  benchmark: RUI test set, email scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_stat_report
  defence_id: stat_gfm
  benchmark: RUI test set, report scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_stat_table
  defence_id: stat_gfm
  benchmark: RUI test set, table scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rand_abstract
  defence_id: rand_gfm
  benchmark: RUI test set, abstract scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rand_code
  defence_id: rand_gfm
  benchmark: RUI test set, code scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rand_email
  defence_id: rand_gfm
  benchmark: RUI test set, email scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rand_report
  defence_id: rand_gfm
  benchmark: RUI test set, report scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rand_table
  defence_id: rand_gfm
  benchmark: RUI test set, table scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rui_abstract
  defence_id: rui_gfm
  benchmark: RUI test set, abstract scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rui_code
  defence_id: rui_gfm
  benchmark: RUI test set, code scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rui_email
  defence_id: rui_gfm
  benchmark: RUI test set, email scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rui_report
  defence_id: rui_gfm
  benchmark: RUI test set, report scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_split_gfm_rui_table
  defence_id: rui_gfm
  benchmark: RUI test set, table scenario, split or encoded injection
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_none_abstract
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, abstract scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_spoof_gfm_none_code
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, code scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_spoof_gfm_none_email
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, email scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_spoof_gfm_none_report
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, report scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_spoof_gfm_none_table
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, table scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_spoof_gfm_stat_abstract
  defence_id: stat_gfm
  benchmark: RUI test set, abstract scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_stat_code
  defence_id: stat_gfm
  benchmark: RUI test set, code scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_stat_email
  defence_id: stat_gfm
  benchmark: RUI test set, email scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_stat_report
  defence_id: stat_gfm
  benchmark: RUI test set, report scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_stat_table
  defence_id: stat_gfm
  benchmark: RUI test set, table scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rand_abstract
  defence_id: rand_gfm
  benchmark: RUI test set, abstract scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rand_code
  defence_id: rand_gfm
  benchmark: RUI test set, code scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rand_email
  defence_id: rand_gfm
  benchmark: RUI test set, email scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rand_report
  defence_id: rand_gfm
  benchmark: RUI test set, report scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rand_table
  defence_id: rand_gfm
  benchmark: RUI test set, table scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rui_abstract
  defence_id: rui_gfm
  benchmark: RUI test set, abstract scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rui_code
  defence_id: rui_gfm
  benchmark: RUI test set, code scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rui_email
  defence_id: rui_gfm
  benchmark: RUI test set, email scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rui_report
  defence_id: rui_gfm
  benchmark: RUI test set, report scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof_gfm_rui_table
  defence_id: rui_gfm
  benchmark: RUI test set, table scenario, delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
- id: ev_spoof5_gfm_none_abstract
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, abstract scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
  baseline_kind: off_the_shelf
- id: ev_spoof5_gfm_none_code
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, code scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
  baseline_kind: off_the_shelf
- id: ev_spoof5_gfm_none_email
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, email scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
  baseline_kind: off_the_shelf
- id: ev_spoof5_gfm_none_report
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, report scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
  baseline_kind: off_the_shelf
- id: ev_spoof5_gfm_none_table
  defence_id: none:Gemini 3 Flash (medium thinking)
  benchmark: RUI test set, table scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
  baseline_kind: off_the_shelf
- id: ev_spoof5_gfm_stat_abstract
  defence_id: stat_gfm
  benchmark: RUI test set, abstract scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_stat_code
  defence_id: stat_gfm
  benchmark: RUI test set, code scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_stat_email
  defence_id: stat_gfm
  benchmark: RUI test set, email scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_stat_report
  defence_id: stat_gfm
  benchmark: RUI test set, report scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_stat_table
  defence_id: stat_gfm
  benchmark: RUI test set, table scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rand_abstract
  defence_id: rand_gfm
  benchmark: RUI test set, abstract scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rand_code
  defence_id: rand_gfm
  benchmark: RUI test set, code scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rand_email
  defence_id: rand_gfm
  benchmark: RUI test set, email scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rand_report
  defence_id: rand_gfm
  benchmark: RUI test set, report scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rand_table
  defence_id: rand_gfm
  benchmark: RUI test set, table scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rui_abstract
  defence_id: rui_gfm
  benchmark: RUI test set, abstract scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rui_code
  defence_id: rui_gfm
  benchmark: RUI test set, code scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rui_email
  defence_id: rui_gfm
  benchmark: RUI test set, email scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rui_report
  defence_id: rui_gfm
  benchmark: RUI test set, report scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_spoof5_gfm_rui_table
  defence_id: rui_gfm
  benchmark: RUI test set, table scenario, five-turn delimitation spoofing
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 18
  decoding: not reported
  attacker_budget: attacker sees the full history; success if any of five turns is compromised
- id: ev_x_gfl_none_abstract
  defence_id: none:Gemini 3 Flash (low thinking)
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gfl_none_code
  defence_id: none:Gemini 3 Flash (low thinking)
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gfl_none_email
  defence_id: none:Gemini 3 Flash (low thinking)
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gfl_none_report
  defence_id: none:Gemini 3 Flash (low thinking)
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gfl_none_table
  defence_id: none:Gemini 3 Flash (low thinking)
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gfl_stat_abstract
  defence_id: stat_gfl
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_stat_code
  defence_id: stat_gfl
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_stat_email
  defence_id: stat_gfl
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_stat_report
  defence_id: stat_gfl
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_stat_table
  defence_id: stat_gfl
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rand_abstract
  defence_id: rand_gfl
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rand_code
  defence_id: rand_gfl
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rand_email
  defence_id: rand_gfl
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rand_report
  defence_id: rand_gfl
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rand_table
  defence_id: rand_gfl
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rui_abstract
  defence_id: rui_gfl
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rui_code
  defence_id: rui_gfl
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rui_email
  defence_id: rui_gfl
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rui_report
  defence_id: rui_gfl
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gfl_rui_table
  defence_id: rui_gfl
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_none_abstract
  defence_id: none:Gemini 3.1 Pro (high thinking)
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gp_none_code
  defence_id: none:Gemini 3.1 Pro (high thinking)
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gp_none_email
  defence_id: none:Gemini 3.1 Pro (high thinking)
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gp_none_report
  defence_id: none:Gemini 3.1 Pro (high thinking)
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gp_none_table
  defence_id: none:Gemini 3.1 Pro (high thinking)
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_gp_stat_abstract
  defence_id: stat_gp
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_stat_code
  defence_id: stat_gp
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_stat_email
  defence_id: stat_gp
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_stat_report
  defence_id: stat_gp
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_stat_table
  defence_id: stat_gp
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rand_abstract
  defence_id: rand_gp
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rand_code
  defence_id: rand_gp
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rand_email
  defence_id: rand_gp
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rand_report
  defence_id: rand_gp
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rand_table
  defence_id: rand_gp
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rui_abstract
  defence_id: rui_gp
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rui_code
  defence_id: rui_gp
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rui_email
  defence_id: rui_gp
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rui_report
  defence_id: rui_gp
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_gp_rui_table
  defence_id: rui_gp
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_none_abstract
  defence_id: none:GLM-5 (high thinking)
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_glm_none_code
  defence_id: none:GLM-5 (high thinking)
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_glm_none_email
  defence_id: none:GLM-5 (high thinking)
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_glm_none_report
  defence_id: none:GLM-5 (high thinking)
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_glm_none_table
  defence_id: none:GLM-5 (high thinking)
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_glm_stat_abstract
  defence_id: stat_glm
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_stat_code
  defence_id: stat_glm
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_stat_email
  defence_id: stat_glm
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_stat_report
  defence_id: stat_glm
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_stat_table
  defence_id: stat_glm
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rand_abstract
  defence_id: rand_glm
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rand_code
  defence_id: rand_glm
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rand_email
  defence_id: rand_glm
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rand_report
  defence_id: rand_glm
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rand_table
  defence_id: rand_glm
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rui_abstract
  defence_id: rui_glm
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rui_code
  defence_id: rui_glm
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rui_email
  defence_id: rui_glm
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rui_report
  defence_id: rui_glm
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_glm_rui_table
  defence_id: rui_glm
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_none_abstract
  defence_id: none:GPT-5.4 (high thinking)
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_g54_none_code
  defence_id: none:GPT-5.4 (high thinking)
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_g54_none_email
  defence_id: none:GPT-5.4 (high thinking)
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_g54_none_report
  defence_id: none:GPT-5.4 (high thinking)
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_g54_none_table
  defence_id: none:GPT-5.4 (high thinking)
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_x_g54_stat_abstract
  defence_id: stat_g54
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_stat_code
  defence_id: stat_g54
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_stat_email
  defence_id: stat_g54
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_stat_report
  defence_id: stat_g54
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_stat_table
  defence_id: stat_g54
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rand_abstract
  defence_id: rand_g54
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rand_code
  defence_id: rand_g54
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rand_email
  defence_id: rand_g54
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rand_report
  defence_id: rand_g54
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rand_table
  defence_id: rand_g54
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rui_abstract
  defence_id: rui_g54
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rui_code
  defence_id: rui_g54
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rui_email
  defence_id: rui_g54
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rui_report
  defence_id: rui_g54
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_x_g54_rui_table
  defence_id: rui_g54
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 21
  decoding: not reported
- id: ev_abl_nopc_abstract
  defence_id: nopc_gfm
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
- id: ev_abl_nopc_code
  defence_id: nopc_gfm
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
- id: ev_abl_nopc_email
  defence_id: nopc_gfm
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
- id: ev_abl_nopc_report
  defence_id: nopc_gfm
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
- id: ev_abl_nopc_table
  defence_id: nopc_gfm
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
- id: ev_abl_noeai_abstract
  defence_id: noeai_gfm
  benchmark: RUI test set, abstract scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
- id: ev_abl_noeai_code
  defence_id: noeai_gfm
  benchmark: RUI test set, code scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
- id: ev_abl_noeai_email
  defence_id: noeai_gfm
  benchmark: RUI test set, email scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
- id: ev_abl_noeai_report
  defence_id: noeai_gfm
  benchmark: RUI test set, report scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
- id: ev_abl_noeai_table
  defence_id: noeai_gfm
  benchmark: RUI test set, table scenario, direct context-aware injection (written with knowledge of the task context)
  attacker_type: adaptive
  attacker_access: grey_box
  judge: attack goal achieved in the response
  layout_same_as_defence: 'yes'
  measured_by: rui_preprints2026
  page: 22
  decoding: not reported
results:
- id: r_dir_gfm_none_abstract
  evaluation_id: ev_dir_gfm_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_none_code
  evaluation_id: ev_dir_gfm_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_none_email
  evaluation_id: ev_dir_gfm_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_none_report
  evaluation_id: ev_dir_gfm_none_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_none_table
  evaluation_id: ev_dir_gfm_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_stat_abstract
  evaluation_id: ev_dir_gfm_stat_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '71.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_stat_code
  evaluation_id: ev_dir_gfm_stat_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '33.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_stat_email
  evaluation_id: ev_dir_gfm_stat_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '27.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_stat_report
  evaluation_id: ev_dir_gfm_stat_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '86.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_stat_table
  evaluation_id: ev_dir_gfm_stat_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '82.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rand_abstract
  evaluation_id: ev_dir_gfm_rand_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '62.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rand_code
  evaluation_id: ev_dir_gfm_rand_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '32.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rand_email
  evaluation_id: ev_dir_gfm_rand_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '23.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rand_report
  evaluation_id: ev_dir_gfm_rand_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '79.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rand_table
  evaluation_id: ev_dir_gfm_rand_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '67.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rui_abstract
  evaluation_id: ev_dir_gfm_rui_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rui_code
  evaluation_id: ev_dir_gfm_rui_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rui_email
  evaluation_id: ev_dir_gfm_rui_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rui_report
  evaluation_id: ev_dir_gfm_rui_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_dir_gfm_rui_table
  evaluation_id: ev_dir_gfm_rui_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
- id: r_bpp_stat_abstract
  evaluation_id: ev_bpp_stat_abstract
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '86.9'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_stat_code
  evaluation_id: ev_bpp_stat_code
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '39.5'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_stat_email
  evaluation_id: ev_bpp_stat_email
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '71.8'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_stat_report
  evaluation_id: ev_bpp_stat_report
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '97.1'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_stat_table
  evaluation_id: ev_bpp_stat_table
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '64.9'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rand_abstract
  evaluation_id: ev_bpp_rand_abstract
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '89.6'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rand_code
  evaluation_id: ev_bpp_rand_code
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '68.9'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rand_email
  evaluation_id: ev_bpp_rand_email
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '58.9'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rand_report
  evaluation_id: ev_bpp_rand_report
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '79.2'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rand_table
  evaluation_id: ev_bpp_rand_table
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '49.1'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rui_abstract
  evaluation_id: ev_bpp_rui_abstract
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '98.9'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rui_code
  evaluation_id: ev_bpp_rui_code
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '88.8'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rui_email
  evaluation_id: ev_bpp_rui_email
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '93.1'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rui_report
  evaluation_id: ev_bpp_rui_report
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '99.4'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_bpp_rui_table
  evaluation_id: ev_bpp_rui_table
  quantity: R2attack
  coverage: proxy
  metric_as_printed: benign performance preservation (BPP)
  value: '99.0'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 2
  reference: the undefended model's response
- id: r_split_gfm_none_abstract
  evaluation_id: ev_split_gfm_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_none_code
  evaluation_id: ev_split_gfm_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_none_email
  evaluation_id: ev_split_gfm_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_none_report
  evaluation_id: ev_split_gfm_none_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_none_table
  evaluation_id: ev_split_gfm_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_stat_abstract
  evaluation_id: ev_split_gfm_stat_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '12.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_stat_code
  evaluation_id: ev_split_gfm_stat_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_stat_email
  evaluation_id: ev_split_gfm_stat_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_stat_report
  evaluation_id: ev_split_gfm_stat_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '29.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_stat_table
  evaluation_id: ev_split_gfm_stat_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rand_abstract
  evaluation_id: ev_split_gfm_rand_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rand_code
  evaluation_id: ev_split_gfm_rand_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rand_email
  evaluation_id: ev_split_gfm_rand_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rand_report
  evaluation_id: ev_split_gfm_rand_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '28.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rand_table
  evaluation_id: ev_split_gfm_rand_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rui_abstract
  evaluation_id: ev_split_gfm_rui_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rui_code
  evaluation_id: ev_split_gfm_rui_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rui_email
  evaluation_id: ev_split_gfm_rui_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rui_report
  evaluation_id: ev_split_gfm_rui_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_split_gfm_rui_table
  evaluation_id: ev_split_gfm_rui_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, split or encoded injection
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_none_abstract
  evaluation_id: ev_spoof_gfm_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_none_code
  evaluation_id: ev_spoof_gfm_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_none_email
  evaluation_id: ev_spoof_gfm_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_none_report
  evaluation_id: ev_spoof_gfm_none_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_none_table
  evaluation_id: ev_spoof_gfm_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_stat_abstract
  evaluation_id: ev_spoof_gfm_stat_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_stat_code
  evaluation_id: ev_spoof_gfm_stat_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_stat_email
  evaluation_id: ev_spoof_gfm_stat_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_stat_report
  evaluation_id: ev_spoof_gfm_stat_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_stat_table
  evaluation_id: ev_spoof_gfm_stat_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rand_abstract
  evaluation_id: ev_spoof_gfm_rand_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rand_code
  evaluation_id: ev_spoof_gfm_rand_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rand_email
  evaluation_id: ev_spoof_gfm_rand_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rand_report
  evaluation_id: ev_spoof_gfm_rand_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rand_table
  evaluation_id: ev_spoof_gfm_rand_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rui_abstract
  evaluation_id: ev_spoof_gfm_rui_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rui_code
  evaluation_id: ev_spoof_gfm_rui_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rui_email
  evaluation_id: ev_spoof_gfm_rui_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rui_report
  evaluation_id: ev_spoof_gfm_rui_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof_gfm_rui_table
  evaluation_id: ev_spoof_gfm_rui_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_none_abstract
  evaluation_id: ev_spoof5_gfm_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_none_code
  evaluation_id: ev_spoof5_gfm_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_none_email
  evaluation_id: ev_spoof5_gfm_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_none_report
  evaluation_id: ev_spoof5_gfm_none_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_none_table
  evaluation_id: ev_spoof5_gfm_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_stat_abstract
  evaluation_id: ev_spoof5_gfm_stat_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_stat_code
  evaluation_id: ev_spoof5_gfm_stat_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_stat_email
  evaluation_id: ev_spoof5_gfm_stat_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_stat_report
  evaluation_id: ev_spoof5_gfm_stat_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_stat_table
  evaluation_id: ev_spoof5_gfm_stat_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rand_abstract
  evaluation_id: ev_spoof5_gfm_rand_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '1.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rand_code
  evaluation_id: ev_spoof5_gfm_rand_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rand_email
  evaluation_id: ev_spoof5_gfm_rand_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rand_report
  evaluation_id: ev_spoof5_gfm_rand_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '3.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rand_table
  evaluation_id: ev_spoof5_gfm_rand_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '2.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rui_abstract
  evaluation_id: ev_spoof5_gfm_rui_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rui_code
  evaluation_id: ev_spoof5_gfm_rui_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rui_email
  evaluation_id: ev_spoof5_gfm_rui_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rui_report
  evaluation_id: ev_spoof5_gfm_rui_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_spoof5_gfm_rui_table
  evaluation_id: ev_spoof5_gfm_rui_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, five-turn delimitation spoofing
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
- id: r_x_gfl_none_abstract
  evaluation_id: ev_x_gfl_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_none_code
  evaluation_id: ev_x_gfl_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_none_email
  evaluation_id: ev_x_gfl_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_none_report
  evaluation_id: ev_x_gfl_none_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_none_table
  evaluation_id: ev_x_gfl_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_stat_abstract
  evaluation_id: ev_x_gfl_stat_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '73.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_stat_code
  evaluation_id: ev_x_gfl_stat_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '50.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_stat_email
  evaluation_id: ev_x_gfl_stat_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '28.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_stat_report
  evaluation_id: ev_x_gfl_stat_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '94.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_stat_table
  evaluation_id: ev_x_gfl_stat_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '84.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rand_abstract
  evaluation_id: ev_x_gfl_rand_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '67.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rand_code
  evaluation_id: ev_x_gfl_rand_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '50.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rand_email
  evaluation_id: ev_x_gfl_rand_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '40.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rand_report
  evaluation_id: ev_x_gfl_rand_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '86.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rand_table
  evaluation_id: ev_x_gfl_rand_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '75.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rui_abstract
  evaluation_id: ev_x_gfl_rui_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '5.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rui_code
  evaluation_id: ev_x_gfl_rui_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rui_email
  evaluation_id: ev_x_gfl_rui_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rui_report
  evaluation_id: ev_x_gfl_rui_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '8.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gfl_rui_table
  evaluation_id: ev_x_gfl_rui_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_none_abstract
  evaluation_id: ev_x_gp_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_none_code
  evaluation_id: ev_x_gp_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_none_email
  evaluation_id: ev_x_gp_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_none_report
  evaluation_id: ev_x_gp_none_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_none_table
  evaluation_id: ev_x_gp_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_stat_abstract
  evaluation_id: ev_x_gp_stat_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_stat_code
  evaluation_id: ev_x_gp_stat_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_stat_email
  evaluation_id: ev_x_gp_stat_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '6.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_stat_report
  evaluation_id: ev_x_gp_stat_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '23.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_stat_table
  evaluation_id: ev_x_gp_stat_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '13.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rand_abstract
  evaluation_id: ev_x_gp_rand_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rand_code
  evaluation_id: ev_x_gp_rand_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rand_email
  evaluation_id: ev_x_gp_rand_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '1.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rand_report
  evaluation_id: ev_x_gp_rand_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '18.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rand_table
  evaluation_id: ev_x_gp_rand_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rui_abstract
  evaluation_id: ev_x_gp_rui_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rui_code
  evaluation_id: ev_x_gp_rui_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rui_email
  evaluation_id: ev_x_gp_rui_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rui_report
  evaluation_id: ev_x_gp_rui_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_gp_rui_table
  evaluation_id: ev_x_gp_rui_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_none_abstract
  evaluation_id: ev_x_glm_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_none_code
  evaluation_id: ev_x_glm_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_none_email
  evaluation_id: ev_x_glm_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_none_report
  evaluation_id: ev_x_glm_none_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_none_table
  evaluation_id: ev_x_glm_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_stat_abstract
  evaluation_id: ev_x_glm_stat_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '23.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_stat_code
  evaluation_id: ev_x_glm_stat_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_stat_email
  evaluation_id: ev_x_glm_stat_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '23.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_stat_report
  evaluation_id: ev_x_glm_stat_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '72.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_stat_table
  evaluation_id: ev_x_glm_stat_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '9.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rand_abstract
  evaluation_id: ev_x_glm_rand_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '11.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rand_code
  evaluation_id: ev_x_glm_rand_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rand_email
  evaluation_id: ev_x_glm_rand_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '5.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rand_report
  evaluation_id: ev_x_glm_rand_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '33.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rand_table
  evaluation_id: ev_x_glm_rand_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rui_abstract
  evaluation_id: ev_x_glm_rui_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rui_code
  evaluation_id: ev_x_glm_rui_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rui_email
  evaluation_id: ev_x_glm_rui_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rui_report
  evaluation_id: ev_x_glm_rui_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_glm_rui_table
  evaluation_id: ev_x_glm_rui_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_none_abstract
  evaluation_id: ev_x_g54_none_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_none_code
  evaluation_id: ev_x_g54_none_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_none_email
  evaluation_id: ev_x_g54_none_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_none_report
  evaluation_id: ev_x_g54_none_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_none_table
  evaluation_id: ev_x_g54_none_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_stat_abstract
  evaluation_id: ev_x_g54_stat_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_stat_code
  evaluation_id: ev_x_g54_stat_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_stat_email
  evaluation_id: ev_x_g54_stat_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '4.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_stat_report
  evaluation_id: ev_x_g54_stat_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '19.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_stat_table
  evaluation_id: ev_x_g54_stat_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '11.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rand_abstract
  evaluation_id: ev_x_g54_rand_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rand_code
  evaluation_id: ev_x_g54_rand_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rand_email
  evaluation_id: ev_x_g54_rand_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rand_report
  evaluation_id: ev_x_g54_rand_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '12.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rand_table
  evaluation_id: ev_x_g54_rand_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rui_abstract
  evaluation_id: ev_x_g54_rui_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rui_code
  evaluation_id: ev_x_g54_rui_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rui_email
  evaluation_id: ev_x_g54_rui_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rui_report
  evaluation_id: ev_x_g54_rui_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_x_g54_rui_table
  evaluation_id: ev_x_g54_rui_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, direct context-aware injection (written with knowledge of the task context)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_abl_nopc_abstract
  evaluation_id: ev_abl_nopc_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '10.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
- id: r_abl_nopc_code
  evaluation_id: ev_abl_nopc_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
- id: r_abl_nopc_email
  evaluation_id: ev_abl_nopc_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '8.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
- id: r_abl_nopc_report
  evaluation_id: ev_abl_nopc_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '14.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
- id: r_abl_nopc_table
  evaluation_id: ev_abl_nopc_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '3.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
- id: r_abl_noeai_abstract
  evaluation_id: ev_abl_noeai_abstract
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '34.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
- id: r_abl_noeai_code
  evaluation_id: ev_abl_noeai_code
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '3.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
- id: r_abl_noeai_email
  evaluation_id: ev_abl_noeai_email
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '28.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
- id: r_abl_noeai_report
  evaluation_id: ev_abl_noeai_report
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '65.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
- id: r_abl_noeai_table
  evaluation_id: ev_abl_noeai_table
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '50.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 22
  location: Table 5
comparisons:
- id: c_dir_gfm_stat_abstract
  left: ev_dir_gfm_stat_abstract
  right: ev_dir_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_stat_code
  left: ev_dir_gfm_stat_code
  right: ev_dir_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_stat_email
  left: ev_dir_gfm_stat_email
  right: ev_dir_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_stat_report
  left: ev_dir_gfm_stat_report
  right: ev_dir_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_stat_table
  left: ev_dir_gfm_stat_table
  right: ev_dir_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rand_abstract
  left: ev_dir_gfm_rand_abstract
  right: ev_dir_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rand_code
  left: ev_dir_gfm_rand_code
  right: ev_dir_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rand_email
  left: ev_dir_gfm_rand_email
  right: ev_dir_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rand_report
  left: ev_dir_gfm_rand_report
  right: ev_dir_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rand_table
  left: ev_dir_gfm_rand_table
  right: ev_dir_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rui_abstract
  left: ev_dir_gfm_rui_abstract
  right: ev_dir_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rui_code
  left: ev_dir_gfm_rui_code
  right: ev_dir_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rui_email
  left: ev_dir_gfm_rui_email
  right: ev_dir_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rui_report
  left: ev_dir_gfm_rui_report
  right: ev_dir_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dir_gfm_rui_table
  left: ev_dir_gfm_rui_table
  right: ev_dir_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_bpp_rui_vs_stat_abstract
  left: ev_bpp_rui_abstract
  right: ev_bpp_stat_abstract
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the stat delimiter
  page: 16
- id: c_bpp_rui_vs_stat_code
  left: ev_bpp_rui_code
  right: ev_bpp_stat_code
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the stat delimiter
  page: 16
- id: c_bpp_rui_vs_stat_email
  left: ev_bpp_rui_email
  right: ev_bpp_stat_email
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the stat delimiter
  page: 16
- id: c_bpp_rui_vs_stat_report
  left: ev_bpp_rui_report
  right: ev_bpp_stat_report
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the stat delimiter
  page: 16
- id: c_bpp_rui_vs_stat_table
  left: ev_bpp_rui_table
  right: ev_bpp_stat_table
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the stat delimiter
  page: 16
- id: c_bpp_rui_vs_rand_abstract
  left: ev_bpp_rui_abstract
  right: ev_bpp_rand_abstract
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the rand delimiter
  page: 16
- id: c_bpp_rui_vs_rand_code
  left: ev_bpp_rui_code
  right: ev_bpp_rand_code
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the rand delimiter
  page: 16
- id: c_bpp_rui_vs_rand_email
  left: ev_bpp_rui_email
  right: ev_bpp_rand_email
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the rand delimiter
  page: 16
- id: c_bpp_rui_vs_rand_report
  left: ev_bpp_rui_report
  right: ev_bpp_rand_report
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the rand delimiter
  page: 16
- id: c_bpp_rui_vs_rand_table
  left: ev_bpp_rui_table
  right: ev_bpp_rand_table
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: RUI against the rand delimiter
  page: 16
- id: c_split_gfm_stat_abstract
  left: ev_split_gfm_stat_abstract
  right: ev_split_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_stat_code
  left: ev_split_gfm_stat_code
  right: ev_split_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_stat_email
  left: ev_split_gfm_stat_email
  right: ev_split_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_stat_report
  left: ev_split_gfm_stat_report
  right: ev_split_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_stat_table
  left: ev_split_gfm_stat_table
  right: ev_split_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rand_abstract
  left: ev_split_gfm_rand_abstract
  right: ev_split_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rand_code
  left: ev_split_gfm_rand_code
  right: ev_split_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rand_email
  left: ev_split_gfm_rand_email
  right: ev_split_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rand_report
  left: ev_split_gfm_rand_report
  right: ev_split_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rand_table
  left: ev_split_gfm_rand_table
  right: ev_split_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rui_abstract
  left: ev_split_gfm_rui_abstract
  right: ev_split_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rui_code
  left: ev_split_gfm_rui_code
  right: ev_split_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rui_email
  left: ev_split_gfm_rui_email
  right: ev_split_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rui_report
  left: ev_split_gfm_rui_report
  right: ev_split_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_split_gfm_rui_table
  left: ev_split_gfm_rui_table
  right: ev_split_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_stat_abstract
  left: ev_spoof_gfm_stat_abstract
  right: ev_spoof_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_stat_code
  left: ev_spoof_gfm_stat_code
  right: ev_spoof_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_stat_email
  left: ev_spoof_gfm_stat_email
  right: ev_spoof_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_stat_report
  left: ev_spoof_gfm_stat_report
  right: ev_spoof_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_stat_table
  left: ev_spoof_gfm_stat_table
  right: ev_spoof_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rand_abstract
  left: ev_spoof_gfm_rand_abstract
  right: ev_spoof_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rand_code
  left: ev_spoof_gfm_rand_code
  right: ev_spoof_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rand_email
  left: ev_spoof_gfm_rand_email
  right: ev_spoof_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rand_report
  left: ev_spoof_gfm_rand_report
  right: ev_spoof_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rand_table
  left: ev_spoof_gfm_rand_table
  right: ev_spoof_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rui_abstract
  left: ev_spoof_gfm_rui_abstract
  right: ev_spoof_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rui_code
  left: ev_spoof_gfm_rui_code
  right: ev_spoof_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rui_email
  left: ev_spoof_gfm_rui_email
  right: ev_spoof_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rui_report
  left: ev_spoof_gfm_rui_report
  right: ev_spoof_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof_gfm_rui_table
  left: ev_spoof_gfm_rui_table
  right: ev_spoof_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_stat_abstract
  left: ev_spoof5_gfm_stat_abstract
  right: ev_spoof5_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_stat_code
  left: ev_spoof5_gfm_stat_code
  right: ev_spoof5_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_stat_email
  left: ev_spoof5_gfm_stat_email
  right: ev_spoof5_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_stat_report
  left: ev_spoof5_gfm_stat_report
  right: ev_spoof5_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_stat_table
  left: ev_spoof5_gfm_stat_table
  right: ev_spoof5_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rand_abstract
  left: ev_spoof5_gfm_rand_abstract
  right: ev_spoof5_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rand_code
  left: ev_spoof5_gfm_rand_code
  right: ev_spoof5_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rand_email
  left: ev_spoof5_gfm_rand_email
  right: ev_spoof5_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rand_report
  left: ev_spoof5_gfm_rand_report
  right: ev_spoof5_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rand_table
  left: ev_spoof5_gfm_rand_table
  right: ev_spoof5_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rui_abstract
  left: ev_spoof5_gfm_rui_abstract
  right: ev_spoof5_gfm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rui_code
  left: ev_spoof5_gfm_rui_code
  right: ev_spoof5_gfm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rui_email
  left: ev_spoof5_gfm_rui_email
  right: ev_spoof5_gfm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rui_report
  left: ev_spoof5_gfm_rui_report
  right: ev_spoof5_gfm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_spoof5_gfm_rui_table
  left: ev_spoof5_gfm_rui_table
  right: ev_spoof5_gfm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_x_gfl_stat_abstract
  left: ev_x_gfl_stat_abstract
  right: ev_x_gfl_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_stat_code
  left: ev_x_gfl_stat_code
  right: ev_x_gfl_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_stat_email
  left: ev_x_gfl_stat_email
  right: ev_x_gfl_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_stat_report
  left: ev_x_gfl_stat_report
  right: ev_x_gfl_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_stat_table
  left: ev_x_gfl_stat_table
  right: ev_x_gfl_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rand_abstract
  left: ev_x_gfl_rand_abstract
  right: ev_x_gfl_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rand_code
  left: ev_x_gfl_rand_code
  right: ev_x_gfl_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rand_email
  left: ev_x_gfl_rand_email
  right: ev_x_gfl_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rand_report
  left: ev_x_gfl_rand_report
  right: ev_x_gfl_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rand_table
  left: ev_x_gfl_rand_table
  right: ev_x_gfl_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rui_abstract
  left: ev_x_gfl_rui_abstract
  right: ev_x_gfl_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rui_code
  left: ev_x_gfl_rui_code
  right: ev_x_gfl_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rui_email
  left: ev_x_gfl_rui_email
  right: ev_x_gfl_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rui_report
  left: ev_x_gfl_rui_report
  right: ev_x_gfl_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gfl_rui_table
  left: ev_x_gfl_rui_table
  right: ev_x_gfl_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_stat_abstract
  left: ev_x_gp_stat_abstract
  right: ev_x_gp_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_stat_code
  left: ev_x_gp_stat_code
  right: ev_x_gp_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_stat_email
  left: ev_x_gp_stat_email
  right: ev_x_gp_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_stat_report
  left: ev_x_gp_stat_report
  right: ev_x_gp_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_stat_table
  left: ev_x_gp_stat_table
  right: ev_x_gp_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rand_abstract
  left: ev_x_gp_rand_abstract
  right: ev_x_gp_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rand_code
  left: ev_x_gp_rand_code
  right: ev_x_gp_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rand_email
  left: ev_x_gp_rand_email
  right: ev_x_gp_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rand_report
  left: ev_x_gp_rand_report
  right: ev_x_gp_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rand_table
  left: ev_x_gp_rand_table
  right: ev_x_gp_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rui_abstract
  left: ev_x_gp_rui_abstract
  right: ev_x_gp_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rui_code
  left: ev_x_gp_rui_code
  right: ev_x_gp_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rui_email
  left: ev_x_gp_rui_email
  right: ev_x_gp_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rui_report
  left: ev_x_gp_rui_report
  right: ev_x_gp_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_gp_rui_table
  left: ev_x_gp_rui_table
  right: ev_x_gp_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_stat_abstract
  left: ev_x_glm_stat_abstract
  right: ev_x_glm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_stat_code
  left: ev_x_glm_stat_code
  right: ev_x_glm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_stat_email
  left: ev_x_glm_stat_email
  right: ev_x_glm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_stat_report
  left: ev_x_glm_stat_report
  right: ev_x_glm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_stat_table
  left: ev_x_glm_stat_table
  right: ev_x_glm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rand_abstract
  left: ev_x_glm_rand_abstract
  right: ev_x_glm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rand_code
  left: ev_x_glm_rand_code
  right: ev_x_glm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rand_email
  left: ev_x_glm_rand_email
  right: ev_x_glm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rand_report
  left: ev_x_glm_rand_report
  right: ev_x_glm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rand_table
  left: ev_x_glm_rand_table
  right: ev_x_glm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rui_abstract
  left: ev_x_glm_rui_abstract
  right: ev_x_glm_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rui_code
  left: ev_x_glm_rui_code
  right: ev_x_glm_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rui_email
  left: ev_x_glm_rui_email
  right: ev_x_glm_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rui_report
  left: ev_x_glm_rui_report
  right: ev_x_glm_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_glm_rui_table
  left: ev_x_glm_rui_table
  right: ev_x_glm_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_stat_abstract
  left: ev_x_g54_stat_abstract
  right: ev_x_g54_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_stat_code
  left: ev_x_g54_stat_code
  right: ev_x_g54_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_stat_email
  left: ev_x_g54_stat_email
  right: ev_x_g54_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_stat_report
  left: ev_x_g54_stat_report
  right: ev_x_g54_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_stat_table
  left: ev_x_g54_stat_table
  right: ev_x_g54_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rand_abstract
  left: ev_x_g54_rand_abstract
  right: ev_x_g54_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rand_code
  left: ev_x_g54_rand_code
  right: ev_x_g54_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rand_email
  left: ev_x_g54_rand_email
  right: ev_x_g54_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rand_report
  left: ev_x_g54_rand_report
  right: ev_x_g54_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rand_table
  left: ev_x_g54_rand_table
  right: ev_x_g54_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rui_abstract
  left: ev_x_g54_rui_abstract
  right: ev_x_g54_none_abstract
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rui_code
  left: ev_x_g54_rui_code
  right: ev_x_g54_none_code
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rui_email
  left: ev_x_g54_rui_email
  right: ev_x_g54_none_email
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rui_report
  left: ev_x_g54_rui_report
  right: ev_x_g54_none_report
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_x_g54_rui_table
  left: ev_x_g54_rui_table
  right: ev_x_g54_none_table
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
- id: c_abl_nopc_abstract
  left: ev_dir_gfm_rui_abstract
  right: ev_abl_nopc_abstract
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
- id: c_abl_nopc_code
  left: ev_dir_gfm_rui_code
  right: ev_abl_nopc_code
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
- id: c_abl_nopc_email
  left: ev_dir_gfm_rui_email
  right: ev_abl_nopc_email
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
- id: c_abl_nopc_report
  left: ev_dir_gfm_rui_report
  right: ev_abl_nopc_report
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
- id: c_abl_nopc_table
  left: ev_dir_gfm_rui_table
  right: ev_abl_nopc_table
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
- id: c_abl_noeai_abstract
  left: ev_dir_gfm_rui_abstract
  right: ev_abl_noeai_abstract
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
- id: c_abl_noeai_code
  left: ev_dir_gfm_rui_code
  right: ev_abl_noeai_code
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
- id: c_abl_noeai_email
  left: ev_dir_gfm_rui_email
  right: ev_abl_noeai_email
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
- id: c_abl_noeai_report
  left: ev_dir_gfm_rui_report
  right: ev_abl_noeai_report
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
- id: c_abl_noeai_table
  left: ev_dir_gfm_rui_table
  right: ev_abl_noeai_table
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: full RUI against the ablated variant
  page: 22
claims:
- id: k1
  text: RUI lowers attack success from 100% to under 8.1% against adaptive attacks while keeping benign performance above 88.8%
  location: abstract
  anchor: reduces the Attack Success Rate (ASR) from 100%
  jobs:
  - R1
  - R2
  supported_by:
  - c_dir_gfm_rui_abstract
  - c_dir_gfm_rui_code
  - c_dir_gfm_rui_email
  - c_dir_gfm_rui_report
  - c_dir_gfm_rui_table
  - c_x_gfl_rui_abstract
  - c_x_gfl_rui_code
  - c_x_gfl_rui_email
  - c_x_gfl_rui_report
  - c_x_gfl_rui_table
  - c_x_gp_rui_abstract
  - c_x_gp_rui_code
  - c_x_gp_rui_email
  - c_x_gp_rui_report
  - c_x_gp_rui_table
  - c_x_glm_rui_abstract
  - c_x_glm_rui_code
  - c_x_glm_rui_email
  - c_x_glm_rui_report
  - c_x_glm_rui_table
  - c_x_g54_rui_abstract
  - c_x_g54_rui_code
  - c_x_g54_rui_email
  - c_x_g54_rui_report
  - c_x_g54_rui_table
  page: 2
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: proxy
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 2 to 5
    R2attack: Table 2 (BPP, judged against the undefended model's output)
    R2clean: not reported
schema_gaps:
- where: report
  note: every undefended configuration is attacked successfully in 100% of cases because the injections were written to succeed; BPP is judged by another LLM relative to the undefended model's output

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04j"
cfg_exclusions = 2
cfg_next = "batch 04k"
IDS = ['longpibench_arxiv2026', 'univdef_arxiv2026', 'explosive_arxiv2026', 'readable_arxiv2026', 'trustmi_arxiv2026', 'pvst_arxiv2026', 'medmas_arxiv2026', 'pipette_arxiv2026', 'cbc_emnlp2026', 'rui_preprints2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 2b. Full-text exclusions found while coding

Each study below was included at title and abstract; its full text shows it does not meet the inclusion rule. The anchor is checked in its PDF, then the decision is written to schema/fulltext_decisions.csv and schema/screening_decisions.csv and carried into reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as notebook 02d did.

In [ ]:
import json
import overlap
importlib.reload(overlap)
EXCL = [{'key': 'arxiv:2609.20188', 'slug': '2609.20188', 'reason': 'survey, position or other', 'anchor': 'The default backend is a deterministic simulated screener', 'note': 'ResumeShield: every result comes from a rule-based simulated screener that by construction obeys directives only outside the fence; no language model is evaluated (live evaluation is left to future work)'}, {'key': 'arxiv:2610.15177', 'slug': '2610.15177', 'reason': 'system level', 'anchor': 'restricts recognized tool-name regions to tools authorized by the task policy', 'note': 'CoDA enforces a task policy of authorised tools, arguments and URLs, written by an oracle or a separate LLM policy selector, as constraints during decoding; this is policy enforcement like a tool allowlist'}]
FT_FIELDS = ["key", "slug", "decision", "reason", "basis", "anchor", "note"]
SD_FIELDS = ["key", "decision", "reason", "stage", "access_status", "note", "title"]
CHECK_FIELDS = ["key", "slug", "decision", "reason", "basis", "page", "status", "anchor", "sha256", "note"]
ft = search.read_csv(paths.SCHEMA / "fulltext_decisions.csv")
sd = search.read_csv(paths.SCHEMA / "screening_decisions.csv")
fchk = search.read_csv(paths.REPORTS / "fulltext_check.csv")
mlist = search.read_csv(paths.REPORTS / "defences.csv")
for e in EXCL:
    m = next((x for x in mlist if x["slug"] == e["slug"]), None)
    pdf = paths.DATA_DEFENCES / f"{e['slug']}.pdf"
    if m is None or not pdf.exists():
        problems.append(f"exclusion {e['slug']}: not in reports/defences.csv or PDF missing"); continue
    sha = corpus.sha256(pdf)
    if sha != m.get("sha256"):
        problems.append(f"exclusion {e['slug']}: PDF differs from the one registered")
    if e["reason"] not in search.EXCLUSION_REASONS:
        problems.append(f"exclusion {e['slug']}: reason not in the fixed list")
    pages = overlap.pdf_page_texts(pdf)
    a = overlap.norm_text(e["anchor"])
    hits = [i + 1 for i, t in enumerate(pages) if a in t]
    status = "found" if hits else "anchor not found"
    if not hits:
        problems.append(f"exclusion {e['slug']}: anchor not found in the PDF")
    row = {"key": e["key"], "slug": e["slug"], "decision": "exclude", "reason": e["reason"], "basis": "full text", "anchor": e["anchor"], "note": e["note"]}
    ft = [r for r in ft if r["key"] != e["key"]] + [row]
    fchk = [r for r in fchk if r["key"] != e["key"]] + [dict(row, page=str(hits[0]) if hits else "", status=status, sha256=sha)]
    n = 0
    for r in sd:
        if r["key"] == e["key"]:
            r.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"]); n += 1
    if n != 1:
        problems.append(f"exclusion {e['slug']}: {n} rows for {e['key']} in schema/screening_decisions.csv")
    m.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"])
    print(f"{e['slug']:<12} excluded at full text ({e['reason']}); anchor {status}{' on page ' + str(hits[0]) if hits else ''}")
def write_lf(path, rows, fields):              # the schema files were written with plain newlines; keep them that way
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore", lineterminator="\n")
        w.writeheader()
        w.writerows({k: r.get(k) or "" for k in fields} for r in rows)
write_lf(paths.SCHEMA / "fulltext_decisions.csv", ft, FT_FIELDS)
write_lf(paths.SCHEMA / "screening_decisions.csv", sd, SD_FIELDS)
search.write_csv(paths.REPORTS / "fulltext_check.csv", fchk, CHECK_FIELDS)
search.write_csv(paths.REPORTS / "defences.csv", mlist, list(mlist[0].keys()))
rows = search.read_csv(paths.REPORTS / "screening.csv")
rows, more = search.apply_decisions(rows, sd)
problems += more
search.write_csv(paths.REPORTS / "screening.csv", rows, search.SCREEN_FIELDS)
prisma = json.loads((paths.REPORTS / "prisma_counts.json").read_text())
prisma.update(search.prisma_counts(search.read_csv(paths.REPORTS / "search_log.csv"), rows))
prisma["fulltext_anchors_found"] = sum(1 for r in fchk if r["status"].startswith("found"))
(paths.REPORTS / "prisma_counts.json").write_text(json.dumps(prisma, indent=1))
print(f"included now: {prisma['included']}; full text assessed: {prisma['full_text_assessed']}; "
      f"full-text exclusions: {json.dumps(prisma['excluded_full_text_by_reason'])}")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept, third_chart = 0, []
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                    third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                else:
                    probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third + third_chart]
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "chart_values_third_extraction": len(third_chart), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept} ({len(third_chart)} via the third extraction); anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in search.read_csv(paths.REPORTS / "defences.csv") if m.get("decision") == "include")   # read again: Stage 2b may change it
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked",
                                              "chart_values_third_extraction", "anchors_checked", "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values ({tot['chart_values_third_extraction']} of them found only in the third extraction) "
         f"and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04j extraction batch j: ten included studies coded from their PDFs and two full-text exclusions, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)